# Tool-Calling Baseline for Clinical Calculators

Baseline in which the model selects a calculator and extracts its parameters, and the calculator's Python implementation is executed to compute the result. The notebook is generic: set the **dataset** and the **model** in the Configuration cell.

**Requirements:** a CUDA GPU; Python packages listed in `requirements.txt`; the dataset CSV and the calculator implementations with `name_to_python.json` referenced in the Configuration cell.

## Evaluation protocol

Part A is identical in all six notebooks of this release (one MCP notebook and five baseline notebooks). Part B lists what is specific to this pipeline. Both parts describe what the code in this notebook does, so that the numbers in the paper can be checked against it.

### A. Common to all pipelines

**Notation.** N is the number of samples after loading the dataset (rows with missing required fields are dropped). S is the set of samples whose selected calculator equals the ground-truth calculator. An *entity* is one ground-truth input variable, i.e. one key of `Relevant Entities`. S' is the set of samples in S that are scored in the extraction stage, and D the set of samples whose final answer is scored (Part B states both).

**Metrics.**

| Metric | Formula |
|---|---|
| CSA, calculator selection accuracy | CSA = \|S\| / N |
| PEA, parameter extraction accuracy | PEA = (matched entities over S') / (ground-truth entities over S') |
| CCA, calculation correctness accuracy | CCA = \|C\| / \|D\|, C = samples in D whose final answer is correct |

A final answer is correct when the number lies inside the dataset interval `[Lower Limit, Upper Limit]` (inclusive). The category-wise evaluation applies the same three formulas to the samples of one dataset category (`Category` column). CCA uses different denominators in different pipelines (Part B), so it is not directly comparable across pipelines. **No end-to-end metric is computed.**

**Reproducibility settings.**

| Item | Setting |
|---|---|
| Random seed | 42 (Python `random`, NumPy, PyTorch, CUDA) |
| Decoding | greedy; `do_sample=False` in every `generate` call; no temperature, top-p or top-k |
| Quantization | 4-bit NF4 with double quantization, float16 compute dtype (bitsandbytes) |
| Model and dataset | set only in the Configuration cell; nothing else is hard-coded |
| Inputs given to the model | the patient note and the question; ground-truth entities and answers are never part of a prompt |
| Normalization agents | PHONA-S (selection output) and PHONA-E (entity output) are rule-based normalizers, not language models |

**Error taxonomy.** The eight types follow Wang et al. (2025), *From Scores to Steps*. Wang et al. assign them with an LLM judge; this release uses the following deterministic rules, identical in all notebooks (cell *Error taxonomy*). Errors are attributed independently at each stage (not only the first failing stage).

| Type | Counted | Rule |
|---|---|---|
| Formula misselection / hallucination | per sample | the selected calculator differs from the ground truth (N - \|S\| samples, including unmappable outputs) |
| Missing variable | per unmatched entity of S' | no value (empty / null / absent) for the entity |
| Demographic adjustment failure | per unmatched entity | the entity is age, sex / gender, race / ethnicity, pregnancy or BSA |
| Unit conversion error | per unmatched entity | the predicted / ground-truth ratio is within 3 % of a common conversion factor (lb-kg, in-cm, creatinine, glucose, BUN, cholesterol, calcium, bilirubin, factors of 10 and 1000) or the values are related by the Celsius-Fahrenheit formula |
| Clinical misinterpretation | per unmatched entity | the ground-truth value is boolean or categorical (presence, severity, threshold) |
| Incorrect variable extraction | per unmatched entity | any other unmatched numeric value or date |
| Precision error | per sample with a wrong result | the result is outside the dataset interval but within Wang et al.'s rounding tolerance of the ground truth: +-0.05 for answers with at most one decimal place, +-0.005 for two or more |
| Arithmetic error | per sample with a wrong result | any other wrong or non-numeric result |

The rules are tested in this order for extraction errors: missing, demographic, unit conversion, clinical misinterpretation, incorrect extraction. The counts are a diagnostic aid and not metrics; extraction errors are counted per entity, so the total can exceed N.

### B. This pipeline: Tool Calling

Selection and extraction are prompted without reasoning or examples; computation is done by executing the calculator's Python implementation, not by the model. Generation limits: selection 30 new tokens, extraction 300.

**Stage 1, selection (CSA).** The prompt contains the patient note and the question only; the list of candidate calculators is *not* shown to the model. The free-text answer is mapped to a calculator identifier by PHONA-S: abbreviation expansion and canonicalization, exact and keyword match against `TOOL_NAME_MAP` / `SEMANTIC_TOOL_MAP`, then `difflib` fuzzy matching with cutoff 0.82. An answer that cannot be mapped is `UNKNOWN_TOOL`; a ground-truth calculator that is not in the map becomes `UNKNOWN_GT` and is always counted as wrong. **CSA = |S| / N**; unmappable outputs count as wrong and stay in the denominator.

**Stage 2, extraction (PEA).** Only samples in S are processed. The JSON output is repaired and normalized by PHONA-E (markdown / JSON repair, key canonicalization with `normalize_semantic_entity_key`, unit and value normalization). An entity is matched when the normalized values are equal, or when both are numbers and differ by at most 0.01 (at most 0.5 for height and weight). S' contains the samples in S whose ground-truth entities and prediction can be parsed and whose prediction is not empty; the others are skipped.

The extraction prompt lists the parameters of the selected calculator with their exact key names and accepted values, read from the calculator registry (`name_to_python.json` in `CALCS_DIR`); parameters that are not found should be returned as `null`. For entity matching, ground-truth keys are normalized with the selected calculator taken into account.

**Stage 3, computation (CCA).** The normalized entities are passed to the *format_inputs* step (fuzzy key matching with cutoff 0.8, values wrapped with units) and then to the calculator function (*execute_tool*). If the entity JSON is empty or the call fails, the result is `None` and the sample is wrong; execution errors are saved to `execution_errors.csv`. D = S, so samples whose extraction failed stay in the denominator and CCA is not comparable with MCP's |C| / |E|.


In [ ]:
# Import required libraries and verify runtime setup.

import os, json, re, random, warnings, ast
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 60)
print("Environment Verification")
print("=" * 60)
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
print("=" * 60)
print("Environment setup completed successfully.")
print("=" * 60)


In [ ]:
# Configuration: the only cell that needs to be edited.
import os

DATASET_NAME = '<dataset-name>'                 # label used in output paths, e.g. 'medcalc', 'abacus212', 'abacus409'
DATASET_FILE = 'data/<dataset_file>.csv'        # dataset CSV (see README for the expected columns)
MODEL_ID     = '<hf-model-id>'                  # any Hugging Face chat model, e.g. 'org/model-name'
CALCS_DIR    = 'calculators'                    # folder with the calculator .py files and name_to_python.json
RESULTS_DIR  = f'results/tool_calling_{DATASET_NAME}_' + MODEL_ID.split('/')[-1]

# If a CSV column name does not match the canonical name, override it here:
# {"csv_column_name": "Canonical Name"}
MANUAL_COLUMN_OVERRIDE = {
    # "relevant_entities": "Relevant Entities",
}

os.makedirs(RESULTS_DIR, exist_ok=True)
checks = [(DATASET_FILE, 'Dataset')] + [(CALCS_DIR, 'Calculators')]
for path, label in checks:
    print(('Found   ' if os.path.exists(path) else 'MISSING '), ':', label, '->', path)

In [ ]:
# Load the dataset.

import pandas as pd

data_raw_df = pd.read_csv(DATASET_FILE)

print("=" * 60)
print("Dataset Loaded")
print("=" * 60)
print("Shape:", data_raw_df.shape)
print("Columns:", data_raw_df.columns.tolist())
print("=" * 60)


In [ ]:
# ============================================================
# ============================================================
from difflib import get_close_matches
from collections import defaultdict

CANONICAL_COLUMNS = [
    "Calculator Name", "Category", "Patient Note", "Question",
    "Relevant Entities", "Ground Truth Answer", "Lower Limit", "Upper Limit"
]

COLUMN_ALIASES = {
    "calculator name": "Calculator Name", "calculator": "Calculator Name",
    "calc name": "Calculator Name", "calc": "Calculator Name",
    "category": "Category", "cat": "Category",
    "patient note": "Patient Note", "vignette": "Patient Note", "note": "Patient Note",
    "question": "Question", "clinical question": "Question", "query": "Question",
    "relevant entities": "Relevant Entities", "entities": "Relevant Entities",
    "ground truth answer": "Ground Truth Answer", "ground truth": "Ground Truth Answer",
    "gt": "Ground Truth Answer", "answer": "Ground Truth Answer",
    "lower limit": "Lower Limit", "lower": "Lower Limit", "lb": "Lower Limit",
    "upper limit": "Upper Limit", "upper": "Upper Limit", "ub": "Upper Limit",
}

def normalize_colname(c):
    return re.sub(r"[^a-z0-9]+", " ", str(c).lower()).strip()

def auto_map_columns(df, manual_override=None):
    """Returns (mapping_dict, unmapped_columns_list, dropped_duplicates_list)."""
    manual_override = manual_override or {}
    canon_norm = {normalize_colname(c): c for c in CANONICAL_COLUMNS}

    raw_mapping, unmapped = {}, []
    for col in df.columns:
        if col in manual_override:
            raw_mapping[col] = manual_override[col]
            continue
        norm = normalize_colname(col)
        if norm in COLUMN_ALIASES:
            raw_mapping[col] = COLUMN_ALIASES[norm]
            continue
        match = get_close_matches(norm, canon_norm.keys(), n=1, cutoff=0.75)
        if match:
            raw_mapping[col] = canon_norm[match[0]]
        else:
            unmapped.append(col)

    target_to_sources = defaultdict(list)
    for src, tgt in raw_mapping.items():
        target_to_sources[tgt].append(src)

    mapping, dropped_duplicates = {}, []
    for tgt, sources in target_to_sources.items():
        if len(sources) == 1:
            mapping[sources[0]] = tgt
            continue
        exact = [s for s in sources if s == tgt]
        keep = exact[0] if exact else sources[0]
        mapping[keep] = tgt
        for s in sources:
            if s != keep:
                dropped_duplicates.append((s, tgt, keep))

    return mapping, unmapped, dropped_duplicates

def derive_question_from_note(note):
    """Return the last sentence of the Patient Note that ends with '?'."""
    note = str(note).strip()
    sentences = re.split(r"(?<=[.?!])\s+", note)
    for s in reversed(sentences):
        if s.strip().endswith("?"):
            return s.strip()
    return ""

print("Column auto-mapper + Question derivation functions ready.")


In [ ]:
# Clean and standardize dataset.

# -- Auto-map columns to canonical names. --
col_mapping, unmapped_cols, dropped_duplicates = auto_map_columns(
    data_raw_df, MANUAL_COLUMN_OVERRIDE
)

print("=" * 60)
print("Column Auto-Mapping Report")
print("-" * 60)
for src, dst in col_mapping.items():
    flag = "(override)" if src in MANUAL_COLUMN_OVERRIDE else ""
    print(f"  {src!r:<25} -> {dst!r} {flag}")
if unmapped_cols:
    print(f"\nUnmapped columns (ignored, likely extra/unused): {unmapped_cols}")
if dropped_duplicates:
    print(f"\nDuplicate-target columns skipped (kept the exact-name match instead):")
    for src, tgt, kept in dropped_duplicates:
        print(f"  {src!r} -> {tgt!r} SKIPPED (kept {kept!r} instead)")
print("=" * 60)

renamed_df = data_raw_df.rename(columns=col_mapping)

# -- Required columns check. --
REQUIRED_COLS = [
    "Calculator Name", "Category", "Patient Note",
    "Relevant Entities", "Ground Truth Answer", "Lower Limit", "Upper Limit"
]  # "Question" is not required: it is derived below.

missing_required = [c for c in REQUIRED_COLS if c not in renamed_df.columns]
if missing_required:
    raise ValueError(
        f"Required columns missing after auto-mapping: {missing_required}. "
        f"Fix this with MANUAL_COLUMN_OVERRIDE."
    )

# -- Derive Question from Patient Note when it is missing or blank. --
if "Question" not in renamed_df.columns:
    renamed_df["Question"] = ""

mask_missing_q = renamed_df["Question"].isna() | (
    renamed_df["Question"].astype(str).str.strip() == ""
)
derived_count = int(mask_missing_q.sum())
renamed_df.loc[mask_missing_q, "Question"] = (
    renamed_df.loc[mask_missing_q, "Patient Note"].apply(derive_question_from_note)
)

still_blank = int((renamed_df["Question"].astype(str).str.strip() == "").sum())
print(f"Question auto-derived for {derived_count} rows "
      f"({still_blank} still blank -- no \'?\' sentence found in Patient Note).")

# -- Dropna report (before dropping). --
before_n = len(renamed_df)
null_report = renamed_df[REQUIRED_COLS].isna().sum()
print("\nNull counts per required column (pre-dropna):")
print(null_report[null_report > 0] if null_report.sum() > 0 else "  None -- all clean.")

data_df = renamed_df[CANONICAL_COLUMNS].dropna(
    subset=REQUIRED_COLS
).reset_index(drop=True)

after_n = len(data_df)
print(f"\nRows before dropna: {before_n}  |  after dropna: {after_n}  "
      f"|  dropped: {before_n - after_n}")

dataset_calculators = data_df["Calculator Name"].unique().tolist()

print("=" * 60)
print("Dataset Cleaned")
print("=" * 60)
print("Total Samples:", len(data_df))
print("Unique Calculators:", len(dataset_calculators))
print("Categories:", data_df["Category"].unique().tolist())
print("=" * 60)


In [ ]:
# ============================================================
# Section 4: Complete Tool Name Normalizer
# Includes: TOOL_NAME_MAP + SEMANTIC_TOOL_MAP (all entries)
#           + upgraded normalize_tool_name() with:
#             - CamelCase split
#             - Medical abbreviation expansion
#             - Keyword containment matching
#             - Fuzzy matching (0.82)
# All entries from Steps 11-52 merged into one cell.
# ============================================================

import re
from difflib import get_close_matches

# ── TOOL_NAME_MAP — exact ground-truth calculator name to OMC tool name. ───────
TOOL_NAME_MAP = {
    "APACHE II Score":                                              "apache_ii",
    "Adjusted Body Weight":                                         "adjusted_body_weight",
    "Ideal Body Weight":                                            "ideal_body_weight",
    "Albumin Corrected Anion Gap":                                  "albumin_corrected_anion",
    "Albumin Corrected Delta Gap":                                  "albumin_corrected_delta_gap",
    "Albumin Corrected Delta Ratio":                                "albumin_delta_ratio",
    "Anion Gap":                                                    "anion_gap",
    "Delta Gap":                                                    "delta_gap",
    "Delta Ratio":                                                  "delta_ratio",
    "Body Mass Index (BMI)":                                        "bmi_calculator",
    "Body Surface Area Calculator":                                 "bsa_calculator",
    "CHA2DS2-VASc Score for Atrial Fibrillation Stroke Risk":       "cha2ds2_vasc_score",
    "Revised Cardiac Risk Index for Pre-Operative Risk":            "cardiac_risk_index",
    "Mean Arterial Pressure (MAP)":                                 "mean_arterial_pressure",
    "Child-Pugh Score for Cirrhosis Mortality":                     "child_pugh_score",
    "MELD Na (UNOS/OPTN)":                                         "meldna",
    "Glasgow Coma Score (GCS)":                                     "glasgow_coma_score",
    "Sequential Organ Failure Assessment (SOFA) Score":             "sofa",
    "Charlson Comorbidity Index (CCI)":                             "cci",
    "Wells' Criteria for DVT":                                      "wells_criteria_dvt",
    "Wells' Criteria for Pulmonary Embolism":                       "wells_criteria_pe",
    "CKD-EPI Equations for Glomerular Filtration Rate":             "ckd-epi_2021_creatinine",
    "Creatinine Clearance (Cockcroft-Gault Equation)":              "creatinine_clearance",
    "MDRD GFR Equation":                                            "mdrd_gfr",
    "Fractional Excretion of Sodium (FENa)":                        "compute_fena",
    "CURB-65 Score for Pneumonia Severity":                         "curb_65",
    "Caprini Score for Venous Thromboembolism (2005)":              "caprini_score",
    "Centor Score (Modified/McIsaac) for Strep Pharyngitis":        "centor_score",
    "FeverPAIN Score for Strep Pharyngitis":                        "feverpain",
    "PSI Score: Pneumonia Severity Index for CAP":                  "psi_score",
    "SIRS Criteria":                                                "sirs_criteria",
    "Calcium Correction for Hypoalbuminemia":                       "calcium_correction",
    "LDL Calculated":                                               "ldl_calculated",
    "Serum Osmolality":                                             "sOsm",
    "Sodium Correction for Hyperglycemia":                          "sodium_correction_hyperglycemia",
    "HOMA-IR (Homeostatic Model Assessment for Insulin Resistance)":"homa_ir",
    "Free Water Deficit":                                           "free_water_deficit",
    "Glasgow-Blatchford Bleeding Score (GBS)":                      "glasgow_bleeding_score",
    "HAS-BLED Score for Major Bleeding Risk":                       "has_bled_score",
    "HEART Score for Major Cardiac Events":                         "heart_score",
    "Framingham Risk Score for Hard Coronary Heart Disease":         "framingham_risk_score",
    "PERC Rule for Pulmonary Embolism":                             "perc_rule",
    "Fibrosis-4 (FIB-4) Index for Liver Fibrosis":                 "fibrosis_4",
    "Estimated Date of Conception":                                 "estimated_conception_date",
    "Estimated Due Date":                                           "estimated_due_date",
    "Estimated Gestational Age":                                    "estimated_gestational_age",
    "Morphine Milligram Equivalents (MME) Calculator":              "mme",
    "Steroid Conversion Calculator":                                "steroid_conversion_calculator",
    "Maintenance Fluids Calculations":                              "maintenance_fluid_calc",
    "QTc Bazett Calculator":                                        "qt_calculator_bazett",
    "QTc Framingham Calculator":                                    "qt_calculator_framingham",
    "QTc Fridericia Calculator":                                    "qt_calculator_fredericia",
    "QTc Hodges Calculator":                                        "qt_calculator_hodges",
    "QTc Rautaharju Calculator":                                    "qt_calculator_rautaharju",
    "Target weight":                                                "target_weight",
}

# ── Noise words removed from SLM output. ──────────────────────
NOISE_WORDS = [
    "score", "calculator", "tool", "criteria",
    "index", "equation", "assessment", "risk"
]

# ── Medical abbreviation expansion map. ───────────────────────
ABBREVIATION_MAP = {
    "ibw": "ideal body weight",
    "abw": "adjusted body weight",
    "fena": "fractional excretion of sodium",
    "qtc": "qt corrected",
    "gfr": "glomerular filtration rate",
    "bsa": "body surface area",
    "bmi": "body mass index",
    "cci": "charlson comorbidity index",
    "rcri": "revised cardiac risk index",
    "perc": "pulmonary embolism rule out criteria",
    "mme": "morphine milligram equivalents",
    "gcs": "glasgow coma score",
    "gbs": "glasgow blatchford bleeding score",
    "scr": "serum creatinine",
    "na":  "sodium",
}

# ── Complete SEMANTIC_TOOL_MAP (all entries merged). ──────────
SEMANTIC_TOOL_MAP = {
    # Liver
    "meld na": "meldna",
    "model for end stage liver disease sodium": "meldna",
    "child pugh": "child_pugh_score",
    "fibrosis 4": "fibrosis_4",
    "fib 4": "fibrosis_4",
    "fibrotest": "fibrosis_4",
    "fibrosis four": "fibrosis_4",
    # Comorbidity
    "charlson mortality": "cci",
    "charlson comorbidity": "cci",
    "charlson": "cci",
    # Critical care
    "sequential organ failure": "sofa",
    "organ failure": "sofa",
    "sofa": "sofa",
    "apache": "apache_ii",
    "glasgow coma": "glasgow_coma_score",
    # Anthropometric
    "body mass": "bmi_calculator",
    "body mass index": "bmi_calculator",
    "bmi": "bmi_calculator",
    "adjusted body weight": "adjusted_body_weight",
    "ideal body weight": "ideal_body_weight",
    "devine formula": "ideal_body_weight",
    "body surface area": "bsa_calculator",
    "target weight": "target_weight",
    "target body weight": "target_weight",
    # Cardiovascular
    "heart cardiac": "heart_score",
    "heart score": "heart_score",
    "cha2ds2 vasc": "cha2ds2_vasc_score",
    "atrial fibrillation stroke": "cha2ds2_vasc_score",
    "cardiac risk index": "cardiac_risk_index",
    "revised cardiac risk index": "cardiac_risk_index",
    "mean arterial pressure": "mean_arterial_pressure",
    "has bled": "has_bled_score",
    "bleeding risk": "has_bled_score",
    "framingham risk": "framingham_risk_score",
    "perc": "perc_rule",
    "perc rule": "perc_rule",
    "glasgow bleeding": "glasgow_bleeding_score",
    "gbs": "glasgow_bleeding_score",
    # Renal
    "gfr ckd epi": "ckd-epi_2021_creatinine",
    "kidney filtration ckd epi": "ckd-epi_2021_creatinine",
    "ckd epi": "ckd-epi_2021_creatinine",
    "ckd epi creatinine equation": "ckd-epi_2021_creatinine",
    "creatinine clearance": "creatinine_clearance",
    "cockcroft gault": "creatinine_clearance",
    "cockroft gault": "creatinine_clearance",
    "mdrd gfr": "mdrd_gfr",
    "fena": "compute_fena",
    "fractional excretion of sodium": "compute_fena",
    # Infectious disease
    "curb 65": "curb_65",
    "pneumonia severity curb": "curb_65",
    "caprini": "caprini_score",
    "centor": "centor_score",
    "feverpain": "feverpain",
    "psi pneumonia": "psi_score",
    "sirs": "sirs_criteria",
    # DVT/PE
    "wells dvt": "wells_criteria_dvt",
    "wells pulmonary embolism": "wells_criteria_pe",
    "wells score": "wells_criteria_pe",
    "wells pe": "wells_criteria_pe",
    # Laboratory
    "corrected calcium": "calcium_correction",
    "calcium correction": "calcium_correction",
    "ldl cholesterol": "ldl_calculated",
    "ldl": "ldl_calculated",
    "friedewald formula": "ldl_calculated",
    "serum osmolality": "sOsm",
    "osmolality": "sOsm",
    "serum osm": "sOsm",
    "anion gap": "anion_gap",
    "delta gap": "delta_gap",
    "delta ratio": "delta_ratio",
    "albumin corrected anion": "albumin_corrected_anion",
    "albumin corrected delta": "albumin_corrected_delta_gap",
    # Metabolic
    "homa ir": "homa_ir",
    "insulin resistance": "homa_ir",
    "hyperglycemia sodium correction": "sodium_correction_hyperglycemia",
    "sodium correction": "sodium_correction_hyperglycemia",
    "hillier": "sodium_correction_hyperglycemia",
    "free water": "free_water_deficit",
    "free water deficit": "free_water_deficit",
    # Obstetric
    "naegele": "estimated_due_date",
    "last menstrual period": "estimated_due_date",
    "due date": "estimated_due_date",
    "estimated due date": "estimated_due_date",
    "estimated conception": "estimated_conception_date",
    "gestational age": "estimated_gestational_age",
    # Medication
    "morphine equivalent": "mme",
    "mme": "mme",
    "opioid conversion": "mme",
    "opioid dose": "mme",
    "morphine milligram equivalent": "mme",
    "steroid conversion": "steroid_conversion_calculator",
    "steroid equivalent": "steroid_conversion_calculator",
    "dose equivalent": "steroid_conversion_calculator",
    "dose equivalence": "steroid_conversion_calculator",
    "dexamethasone to prednisone": "steroid_conversion_calculator",
    "dexamethasone to hydrocortisone": "steroid_conversion_calculator",
    "prednisone to hydrocortisone": "steroid_conversion_calculator",
    # Fluids
    "maintenance fluids": "maintenance_fluid_calc",
    "maintenance fluid": "maintenance_fluid_calc",
    # QT
    "qtc bazett": "qt_calculator_bazett",
    "bazett formula": "qt_calculator_bazett",
    "qtc framingham": "qt_calculator_framingham",
    "qtc fridericia": "qt_calculator_fredericia",
    "fridericia formula": "qt_calculator_fredericia",
    "qtc hodges": "qt_calculator_hodges",
    "hodges formula": "qt_calculator_hodges",
    "qtc rautaharju": "qt_calculator_rautaharju",
    "rautaharju formula": "qt_calculator_rautaharju",
}

# ── Normalizer helper functions. ──────────────────────────────
def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[\-_'():/]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

def remove_noise_words(text):
    return " ".join(w for w in text.split() if w not in NOISE_WORDS)

def split_camel_case(text):
    text = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", text)
    text = re.sub(r"(?<=[A-Z])(?=[A-Z][a-z])", " ", text)
    return text

def normalize_medical_abbreviations(text):
    return " ".join(ABBREVIATION_MAP.get(w, w) for w in text.split())

def canonicalize(raw_text):
    text = split_camel_case(str(raw_text))
    text = clean_text(text)
    text = normalize_medical_abbreviations(text)
    text = remove_noise_words(text)
    return re.sub(r"\s+", " ", text).strip()

def contains_keyword_match(filtered_name):
    for phrase, tool in SEMANTIC_TOOL_MAP.items():
        cp = canonicalize(phrase)
        if cp in filtered_name or filtered_name in cp:
            return tool
    for ds_name, tool in TOOL_NAME_MAP.items():
        cd = canonicalize(ds_name)
        if cd in filtered_name or filtered_name in cd:
            return tool
    return "UNKNOWN_TOOL"

def fuzzy_match_tool_name(filtered_name):
    candidate_map = {}
    for phrase, tool in SEMANTIC_TOOL_MAP.items():
        candidate_map[canonicalize(phrase)] = tool
    for ds_name, tool in TOOL_NAME_MAP.items():
        candidate_map[canonicalize(ds_name)] = tool
    candidates = list(candidate_map.keys())
    matches = get_close_matches(filtered_name, candidates, n=1, cutoff=0.82)
    return candidate_map[matches[0]] if matches else "UNKNOWN_TOOL"

def normalize_tool_name(raw_tool_name):
    filtered = canonicalize(raw_tool_name)
    if filtered in SEMANTIC_TOOL_MAP:
        return SEMANTIC_TOOL_MAP[filtered]
    for ds_name, tool in TOOL_NAME_MAP.items():
        if filtered == canonicalize(ds_name):
            return tool
    kw = contains_keyword_match(filtered)
    if kw != "UNKNOWN_TOOL":
        return kw
    fz = fuzzy_match_tool_name(filtered)
    if fz != "UNKNOWN_TOOL":
        return fz
    return "UNKNOWN_TOOL"

print("=" * 60)
print("Complete Tool Normalizer Ready")
print("TOOL_NAME_MAP entries:", len(TOOL_NAME_MAP))
print("SEMANTIC_TOOL_MAP entries:", len(SEMANTIC_TOOL_MAP))
print("=" * 60)


In [ ]:
# ============================================================
# ============================================================
TOOL_NAME_MAP["NIH Stroke Scale/Score"] = "nihss"
TOOL_NAME_MAP["ARISCAT Score for Postoperative Pulmonary Complications"] = "ariscat_score"
TOOL_NAME_MAP["General Anxiety Disorder-7"] = "gad7"

missing_calcs = [
    c for c in data_df["Calculator Name"].unique()
    if c not in TOOL_NAME_MAP
]
if missing_calcs:
    affected = data_df["Calculator Name"].isin(missing_calcs).sum()
    print("=" * 60)
    print(f"WARNING: {len(missing_calcs)} calculator name(s) not in TOOL_NAME_MAP "
          f"({affected} rows affected -- will resolve to UNKNOWN_TOOL in selection):")
    for c in missing_calcs:
        print("  -", c)
    print("=" * 60)
else:
    print("All Calculator Name values found in TOOL_NAME_MAP. Good.")


In [ ]:
# Load the model with 4-bit NF4 quantization (dependencies: see requirements.txt).
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login

if os.environ.get('HF_TOKEN'):   # only needed for gated models; never store a token in the notebook
    login(token=os.environ['HF_TOKEN'])

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

slm_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
slm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
slm_model.eval()
print(MODEL_ID, "loaded.")

In [ ]:
# ============================================================
# Load OpenMedCalc Tool Registry.
# Maps calculator names → executable .py files + params.
# ============================================================
import sys, importlib
from difflib import get_close_matches

sys.path.insert(0, CALCS_DIR)

with open(os.path.join(CALCS_DIR, "name_to_python.json")) as f:
    NAME_TO_PYTHON = json.load(f)

SKIP_KEYS = {"file path","explanation function","calculator name","type","question"}
CALC_REGISTRY = {}
for key, info in NAME_TO_PYTHON.items():
    calc_name = info.get("calculator name", "")
    CALC_REGISTRY[calc_name] = {
        "file":     info.get("file path", ""),
        "function": info.get("explanation function", ""),
        "params":   {k: v for k, v in info.items() if k not in SKIP_KEYS}
    }

print("=" * 60)
print("OpenMedCalc Tool Registry Loaded")
print("Total Calculators:", len(CALC_REGISTRY))
print("=" * 60)


In [ ]:
# ============================================================
# CALCULATOR_FIELD_SCHEMAS -- verified per-field extraction spec
# for the calculators of this dataset. Every param_key, type, and
# categorical value list below was verified directly against the
# calculator .py source files (not guessed) -- see conversation
# history for the grep/read commands used to confirm each one.
# ============================================================

CALCULATOR_FIELD_SCHEMAS = {

    "cci": [
        {"key": "age", "type": "number", "unit": "years"},
        {"key": "mi", "type": "boolean", "label": "Myocardial infarction"},
        {"key": "chf", "type": "boolean", "label": "Congestive Heart Failure"},
        {"key": "peripheral_vascular_disease", "type": "boolean"},
        {"key": "cva", "type": "boolean", "label": "Cerebrovascular Accident"},
        {"key": "tia", "type": "boolean", "label": "Transient Ischemic Attack history"},
        {"key": "dementia", "type": "boolean"},
        {"key": "copd", "type": "boolean", "label": "Chronic Pulmonary Disease"},
        {"key": "connective_tissue_disease", "type": "boolean"},
        {"key": "peptic_ucler_disease", "type": "boolean", "label": "Peptic ulcer disease"},
        {"key": "hemiplegia", "type": "boolean"},
        {"key": "moderate_to_severe_ckd", "type": "boolean", "label": "Moderate to severe Chronic Kidney Disease"},
        {"key": "leukemia", "type": "boolean"},
        {"key": "lymphoma", "type": "boolean"},
        {"key": "aids", "type": "boolean"},
        {"key": "liver_disease", "type": "categorical", "label": "Liver disease severity",
         "values": ["none", "mild", "moderate to severe"]},
        {"key": "diabetes_mellitus", "type": "categorical",
         "values": ["none or diet-controlled", "uncomplicated", "end-organ damage"]},
        {"key": "solid_tumor", "type": "categorical",
         "values": ["none", "localized", "metastatic"]},
    ],

    "wells_criteria_dvt": [
        {"key": "alternative_to_dvt_diagnosis", "type": "boolean",
         "label": "Alternative diagnosis to DVT as likely or more likely"},
        {"key": "localized_tenderness_on_deep_venuous_system", "type": "boolean",
         "label": "Localized tenderness along the deep venous system"},
        {"key": "previous_dvt_documented", "type": "boolean",
         "label": "Previously documented Deep Vein Thrombosis"},
        {"key": "paralysis_paresis_immobilization_in_lower_extreme", "type": "boolean",
         "label": "Paralysis, paresis, or recent plaster immobilization of the lower extremity"},
        {"key": "leg_swollen", "type": "boolean", "label": "Entire Leg Swollen"},
        {"key": "bedridden_for_atleast_3_days", "type": "boolean", "label": "Bedridden recently >3 days"},
        {"key": "active_cancer", "type": "boolean"},
        {"key": "major_surgery_in_last_12_weeks", "type": "boolean"},
        {"key": "calf_swelling_3cm", "type": "boolean",
         "label": "Calf swelling >3cm compared to the other leg"},
        {"key": "collateral_superficial_veins", "type": "boolean",
         "label": "Collateral (nonvaricose) superficial veins present"},
        {"key": "pitting_edema_on_symptomatic_leg", "type": "boolean",
         "label": "Pitting edema confined to symptomatic leg"},
    ],

    "meldna": [
        {"key": "creatinine", "type": "number", "unit": "mg/dL"},
        {"key": "cvvhd", "type": "boolean",
         "label": "Continuous veno-venous hemodialysis for >=24 hours in the past week"},
        {"key": "dialysis_twice", "type": "boolean",
         "label": "Dialysis at least twice in the past week"},
        {"key": "inr", "type": "number", "unit": None,
         "note": "international normalized ratio -- scalar, NOT a list"},
        {"key": "sodium", "type": "number", "unit": "mEq/L"},
        {"key": "bilirubin", "type": "number", "unit": "mg/dL"},
    ],

    "sofa": [
        {"key": "pao2", "type": "number", "unit": "mm Hg", "label": "PaO2"},
        {"key": "fio2", "type": "number", "unit": "%", "label": "FiO2"},
        {"key": "mechanical_ventilation", "type": "boolean", "label": "On mechanical ventilation"},
        {"key": "cpap", "type": "boolean", "label": "Continuous positive airway pressure"},
        {"key": "platelet_count", "type": "number", "unit": "µL"},
        {"key": "gcs", "type": "number", "unit": None,
         "note": "Glasgow Coma Score, 3-15 -- scalar, NOT a list"},
        {"key": "bilirubin", "type": "number", "unit": "mg/dL"},
        {"key": "hypotension", "type": "boolean"},
        {"key": "sys_bp", "type": "number", "unit": "mm Hg", "label": "Systolic Blood Pressure"},
        {"key": "dia_bp", "type": "number", "unit": "mm Hg", "label": "Diastolic Blood Pressure"},
        {"key": "dopamine", "type": "number", "unit": "mcg/kg/min"},
        {"key": "dobutamine", "type": "number", "unit": "mcg/kg/min"},
        {"key": "norepinephrine", "type": "number", "unit": "mcg/kg/min"},
        {"key": "epinephrine", "type": "number", "unit": "mcg/kg/min"},
        {"key": "creatinine", "type": "number", "unit": "mg/dL"},
        {"key": "urine_output", "type": "number", "unit": "mL/day"},
    ],

    "caprini_score": [
        {"key": "age", "type": "number", "unit": "years"},
        {"key": "sex", "type": "string", "values": ["Male", "Female"]},
        {"key": "surgery_type", "type": "categorical",
         "values": ["none", "minor", "major", "laparoscopic", "arthroscopic",
                    "elective major lower extremity arthroplasty"]},
        {"key": "mobility", "type": "categorical",
         "values": ["normal", "on bed rest", "confined to bed >72 hours"]},
        {"key": "bmi", "type": "number", "unit": "kg/m^2", "label": "Body Mass Index"},
        {"key": "major_surgery_last_month", "type": "boolean"},
        {"key": "chf_last_month", "type": "boolean", "label": "Congestive Heart Failure in the last month"},
        {"key": "sepsis", "type": "boolean", "label": "Sepsis in the last month"},
        {"key": "pneumonia", "type": "boolean", "label": "Pneumonia in the last month"},
        {"key": "immobilizing_plaster_cast", "type": "boolean"},
        {"key": "hip_pelvis_leg_fracture", "type": "boolean", "label": "Hip, pelvis, or leg fracture in the last month"},
        {"key": "stroke_last_month", "type": "boolean"},
        {"key": "multiple_trauma", "type": "boolean", "label": "Multiple trauma in the last month"},
        {"key": "acute_spinal_chord_injury", "type": "boolean",
         "label": "Acute spinal cord injury causing paralysis in the last month"},
        {"key": "varicose_veins", "type": "boolean"},
        {"key": "current_swollen_legs", "type": "boolean"},
        {"key": "current_central_venuous", "type": "boolean", "label": "Current central venous access"},
        {"key": "previous_dvt", "type": "boolean", "label": "Previously documented Deep Vein Thrombosis"},
        {"key": "previous_pe", "type": "boolean", "label": "Previously Documented Pulmonary Embolism"},
        {"key": "family_history_thrombosis", "type": "boolean"},
        {"key": "positive_factor_v", "type": "boolean", "label": "Positive Factor V Leiden"},
        {"key": "positive_prothrombin", "type": "boolean", "label": "Positive prothrombin 20210A"},
        {"key": "serum_homocysteine", "type": "boolean", "label": "Elevated serum homocysteine"},
        {"key": "positive_lupus_anticoagulant", "type": "boolean"},
        {"key": "elevated_anticardiolipin_antibody", "type": "boolean"},
        {"key": "heparin_induced_thrombocytopenia", "type": "boolean"},
        {"key": "congenital_acquired_thrombophilia", "type": "boolean",
         "label": "Other congenital or acquired thrombophilia"},
        {"key": "inflammatory_bowel_disease", "type": "boolean", "label": "History of inflammatory bowel disease"},
        {"key": "acute_myocardial_infarction", "type": "boolean"},
        {"key": "copd", "type": "boolean", "label": "Chronic Obstructive Pulmonary Disease"},
        {"key": "malignancy", "type": "boolean", "label": "Present or previous malignancy"},
    ],

    "has_bled_score": [
        {"key": "age", "type": "number", "unit": "years"},
        {"key": "alcoholic_drinks", "type": "number", "unit": "drinks/week",
         "label": "Number of Alcoholic Drinks Per Week"},
        {"key": "liver_disease_has_bled", "type": "boolean", "label": "Liver disease (HAS-BLED criteria)"},
        {"key": "renal_disease_has_bled", "type": "boolean", "label": "Renal disease (HAS-BLED criteria)"},
        {"key": "hypertension", "type": "boolean"},
        {"key": "stroke", "type": "boolean"},
        {"key": "medications_for_bleeding", "type": "boolean", "label": "Medication usage predisposing to bleeding"},
        {"key": "prior_bleeding", "type": "boolean", "label": "Prior major bleeding or predisposition to bleeding"},
        {"key": "labile_inr", "type": "boolean", "label": "Labile international normalized ratio"},
    ],

    "ariscat_score": [
        {"key": "age", "type": "number", "unit": "years"},
        {"key": "spo2", "type": "number", "unit": "%", "label": "preoperative SpO2"},
        {"key": "respiratory_infection", "type": "boolean", "label": "respiratory infection in the last month"},
        {"key": "preoperative_anemia", "type": "boolean", "label": "preoperative anemia (Hct < 30%)"},
        {"key": "surgical_incision", "type": "categorical",
         "values": ["peripheral", "upper abdominal", "intrathoracic"]},
        {"key": "duration", "type": "categorical", "label": "duration of surgery",
         "values": ["less than 2 hours", "2-3 hours", "more than 3 hours"]},
        {"key": "emergency", "type": "boolean", "label": "emergency procedure"},
    ],

    "nihss": [
        {"key": "loc", "type": "number", "unit": None, "label": "Level of consciousness", "range": "0-3"},
        {"key": "orientation", "type": "number", "unit": None, "label": "Orientation questions", "range": "0-2"},
        {"key": "commands", "type": "number", "unit": None, "range": "0-2"},
        {"key": "gaze", "type": "number", "unit": None, "label": "Best gaze", "range": "0-2"},
        {"key": "visual", "type": "number", "unit": None, "label": "Visual fields", "range": "0-3"},
        {"key": "facial", "type": "number", "unit": None, "label": "Facial palsy", "range": "0-3"},
        {"key": "left_arm", "type": "number", "unit": None, "label": "Left arm motor", "range": "0-4"},
        {"key": "right_arm", "type": "number", "unit": None, "label": "Right arm motor", "range": "0-4"},
        {"key": "left_leg", "type": "number", "unit": None, "label": "Left leg motor", "range": "0-4"},
        {"key": "right_leg", "type": "number", "unit": None, "label": "Right leg motor", "range": "0-4"},
        {"key": "ataxia", "type": "number", "unit": None, "label": "Limb ataxia", "range": "0-2"},
        {"key": "sensory", "type": "number", "unit": None, "range": "0-2"},
        {"key": "language", "type": "number", "unit": None, "range": "0-3"},
        {"key": "dysarthria", "type": "number", "unit": None, "range": "0-2"},
        {"key": "inattention", "type": "number", "unit": None, "label": "Extinction/Inattention", "range": "0-2"},
    ],

    "psi_score": [
        {"key": "heart_rate", "type": "number", "unit": "bpm", "label": "Heart Rate or Pulse"},
        {"key": "pH", "type": "number", "unit": None, "note": "scalar, NOT a list"},
        {"key": "sys_bp", "type": "number", "unit": "mm Hg", "label": "Systolic Blood Pressure"},
        {"key": "bun", "type": "number", "unit": "mg/dL", "label": "Blood Urea Nitrogen"},
        {"key": "renal_disease", "type": "boolean", "label": "Renal disease history"},
        {"key": "cerebrovascular_disease", "type": "boolean", "label": "Cerebrovascular disease history"},
        {"key": "neoplastic_disease", "type": "boolean"},
        {"key": "sodium", "type": "number", "unit": "mEq/L"},
        {"key": "liver_disease", "type": "boolean", "label": "Liver disease history"},
        {"key": "chf", "type": "boolean", "label": "Congestive Heart Failure"},
        {"key": "nursing_home_resident", "type": "boolean"},
        {"key": "partial_pressure_oxygen", "type": "number", "unit": "mm Hg"},
        {"key": "age", "type": "number", "unit": "years"},
        {"key": "respiratory_rate", "type": "number", "unit": "breaths/min"},
        {"key": "sex", "type": "string", "values": ["Male", "Female"]},
        {"key": "hematocrit", "type": "number", "unit": "%"},
        {"key": "glucose", "type": "number", "unit": "mg/dL"},
        {"key": "temperature", "type": "number", "unit": "celsius"},
        {"key": "pleural_effusion", "type": "boolean", "label": "Pleural effusion on x-ray"},
        {"key": "altered_mental_status", "type": "boolean"},
    ],

    "gad7": [
        {"key": "feeling_nervous", "type": "number", "unit": None, "range": "0-3",
         "note": "0=not at all, 1=several days, 2=more than half the days, 3=nearly every day"},
        {"key": "not_able_to_stop_worrying", "type": "number", "unit": None, "range": "0-3"},
        {"key": "worrying_too_much", "type": "number", "unit": None, "range": "0-3"},
        {"key": "trouble_relaxing", "type": "number", "unit": None, "range": "0-3"},
        {"key": "being_restless", "type": "number", "unit": None, "range": "0-3"},
        {"key": "easily_annoyed", "type": "number", "unit": None, "range": "0-3"},
        {"key": "feeling_afraid", "type": "number", "unit": None, "range": "0-3"},
    ],
}

print("=" * 60)
print("CALCULATOR_FIELD_SCHEMAS loaded")
print("Calculators covered:", len(CALCULATOR_FIELD_SCHEMAS))
for slug, fields in CALCULATOR_FIELD_SCHEMAS.items():
    print(f"  {slug:<20} {len(fields)} fields")
print("=" * 60)


In [ ]:
# ============================================================
# Tool Executor — Agent 3 + Agent 4.
# ============================================================
import re, importlib
from difflib import get_close_matches

UNIT_MAP = {
    "age": "years", "weight": "kg", "height": "cm",
    "creatinine": "mg/dL", "sodium": "mEq/L", "potassium": "mEq/L",
    "chloride": "mEq/L", "bicarbonate": "mEq/L", "bun": "mg/dL",
    "glucose": "mg/dL", "albumin": "g/dL", "bilirubin": "mg/dL",
    "hemoglobin": "g/dL", "platelets": "10^9/L", "inr": "ratio",
    "ptt": "seconds", "temperature": "celsius",
    "heart_rate": "beats/min", "systolic_bp": "mm Hg",
    "diastolic_bp": "mm Hg", "spo2": "%", "fio2": "%", "pao2": "mm Hg",
    "platelet_count":          "µL",
    "partial_pressure_oxygen": "mm Hg",
    "sys_bp": "mm Hg", "dia_bp": "mm Hg",
    "dopamine": "mcg/kg/min", "dobutamine": "mcg/kg/min",
    "norepinephrine": "mcg/kg/min", "epinephrine": "mcg/kg/min",
    "urine_output": "mL/day", "hematocrit": "%",
    "respiratory_rate": "breaths/min", "alcoholic_drinks": "drinks/week",
    "bmi": "kg/m^2",
}

SCALAR_KEYS = {"inr", "gcs", "pH"}

SEX_NORM = {
    "male": "Male", "m": "Male", "man": "Male", "Male": "Male",
    "female": "Female", "f": "Female", "woman": "Female", "Female": "Female"
}

BOOLEAN_TRUE = {
    "yes", "true", "present", "positive", "1", "y", "t",
    "confirmed", "known", "history", "active", "current",
    "chronic", "diagnosed", "documented", "reported"
}

BOOLEAN_FALSE = {
    "no", "false", "absent", "negative", "0", "n", "f",
    "none", "not present", "not reported", "denied",
    "normal", "no history", "without"
}

# Fallback global sets -- used only when the calculator ISN'T one of
# the 10 schema-covered ones below (CALCULATOR_FIELD_SCHEMAS takes
# priority when it exists for that calculator).
CATEGORICAL_KEYS_FALLBACK = {"diabetes_mellitus", "liver_disease", "solid_tumor"}

STRICT_BOOLEAN_KEYS_FALLBACK = {
    "mi", "chf", "peripheral_vascular_disease", "cva", "tia",
    "dementia", "copd", "connective_tissue_disease",
    "peptic_ucler_disease", "hemiplegia", "moderate_to_severe_ckd",
    "leukemia", "lymphoma", "aids",
    "active_cancer", "previous_dvt_documented",
    "collateral_superficial_veins", "bedridden_for_atleast_3_days",
    "major_surgery_in_last_12_weeks",
    "localized_tenderness_on_deep_venuous_system",
    "alternative_to_dvt_diagnosis",
}

NIHSS_SUBITEM_KEYS = {
    "loc", "orientation", "commands", "gaze", "visual", "facial",
    "left_arm", "right_arm", "left_leg", "right_leg", "ataxia",
    "sensory", "language", "dysarthria", "inattention",
}

# ── Direct registry map ───────────────────────────────
DIRECT_REGISTRY_MAP = {
    "bmi_calculator":                 "Body Mass Index (BMI)",
    "ckd-epi_2021_creatinine":        "CKD-EPI Equations for Glomerular Filtration Rate",
    "calcium_correction":             "Calcium Correction for Hypoalbuminemia",
    "meldna":                         "MELD Na (UNOS/OPTN)",
    "fibrosis_4":                     "Fibrosis-4 (FIB-4) Index for Liver Fibrosis",
    "homa_ir":                        "HOMA-IR (Homeostatic Model Assessment for Insulin Resistance)",
    "sOsm":                           "Serum Osmolality",
    "free_water_deficit":             "Free Water Deficit",
    "compute_fena":                   "Fractional Excretion of Sodium (FENa)",
    "ldl_calculated":                 "LDL Calculated",
    "albumin_corrected_anion":        "Albumin Corrected Anion Gap",
    "albumin_corrected_delta_gap":    "Albumin Corrected Delta Gap",
    "albumin_delta_ratio":            "Albumin Corrected Delta Ratio",
    "delta_gap":                      "Delta Gap",
    "delta_ratio":                    "Delta Ratio",
    "bsa_calculator":                 "Body Surface Area Calculator",
    "adjusted_body_weight":           "Adjusted Body Weight",
    "glasgow_coma_score":             "Glasgow Coma Score (GCS)",
    "apache_ii":                      "APACHE II Score",
    "sofa":                           "Sequential Organ Failure Assessment (SOFA) Score",
    "caprini_score":                  "Caprini Score for Venous Thromboembolism (2005)",
    "estimated_due_date":             "Estimated Due Date",
    "steroid_conversion_calculator":  "Steroid Conversion Calculator",
    "mme":                            "Morphine Milligram Equivalents (MME) Calculator",
    "estimated_gestational_age":      "Estimated Gestational Age",
    "feverpain":                      "FeverPAIN Score for Strep Pharyngitis",
    "centor_score":                   "Centor Score (Modified/McIsaac) for Strep Pharyngitis",
    "perc_rule":                      "PERC Rule for Pulmonary Embolism",
    "sirs_criteria":                  "SIRS Criteria",
    "has_bled_score":                 "HAS-BLED Score for Major Bleeding Risk",
    "curb_65":                        "CURB-65 Score for Pneumonia Severity",
    "cardiac_risk_index":             "Revised Cardiac Risk Index for Pre-Operative Risk",
    "wells_criteria_dvt":             "Wells' Criteria for DVT",
    "wells_criteria_pe":              "Wells' Criteria for Pulmonary Embolism",
    "cha2ds2_vasc_score":             "CHA2DS2-VASc Score for Atrial Fibrillation Stroke Risk",
    "cci":                            "Charlson Comorbidity Index (CCI)",
    "heart_score":                    "HEART Score for Major Cardiac Events",
    "child_pugh_score":               "Child-Pugh Score for Cirrhosis Mortality",
    "qt_calculator_bazett":           "QTc Bazett Calculator",
    "qt_calculator_hodges":           "QTc Hodges Calculator",
    "qt_calculator_rautaharju":       "QTc Rautaharju Calculator",
    "qt_calculator_fredericia":       "QTc Fridericia Calculator",
    "qt_calculator_framingham":       "QTc Framingham Calculator",
    "framingham_risk_score":          "Framingham Risk Score for Hard Coronary Heart Disease",
    "creatinine_clearance":           "Creatinine Clearance (Cockcroft-Gault Equation)",
    "anion_gap":                      "Anion Gap",
    "mdrd_gfr":                       "MDRD GFR Equation",
    "mean_arterial_pressure":         "Mean Arterial Pressure (MAP)",
    "sodium_correction_hyperglycemia":"Sodium Correction for Hyperglycemia",
    "ideal_body_weight":              "Ideal Body Weight",
    "maintenance_fluid_calc":         "Maintenance Fluids Calculations",
    "ariscat_score":                  "ARISCAT Score for Postoperative Pulmonary Complications",
    "nihss":                          "NIH Stroke Scale (NIHSS)",
    "psi_score":                      "PSI Score: Pneumonia Severity Index for CAP",
    "gad7":                           "General Anxiety Disorder-7 (GAD-7)",
    "arterial_blood_gas":             "Arterial Blood Gas Analyzer",
    "ciwa_ar":                        "CIWA-Ar for Alcohol Withdrawal",
    "grace_acs":                      "GRACE ACS Risk and Mortality Calculator",
    "gupta_perioperative":            "Gupta Perioperative Risk for Myocardial Infarction or Cardiac Arrest",
    "pecarn_head_injury":             "PECARN Pediatric Head Injury/Trauma Algorithm",
    "pregnancy_due_dates":            "Pregnancy Due Dates Calculator",
    "stop_bang":                      "STOP-BANG Score for Obstructive Sleep Apnea",
    "ascvd_risk":                     "Atherosclerotic Cardiovascular Disease (ASCVD) Risk Algorithm",
    "phq9":                           "Patient Health Questionnaire-9 (PHQ-9)",
    "sirs_criteria":                  "SIRS Criteria",
}

def get_registry_name(selected_tool):
    if selected_tool in DIRECT_REGISTRY_MAP:
        return DIRECT_REGISTRY_MAP[selected_tool]
    if selected_tool in CALC_REGISTRY:
        return selected_tool
    tool_lower = selected_tool.lower().replace("_", " ")
    for name in CALC_REGISTRY.keys():
        if tool_lower in name.lower() or name.lower() in tool_lower:
            return name
    names       = list(CALC_REGISTRY.keys())
    names_lower = [n.lower() for n in names]
    matches = get_close_matches(tool_lower, names_lower, n=1, cutoff=0.5)
    if matches:
        return names[names_lower.index(matches[0])]
    return None


def _schema_field_type(selected_tool, key):
    """look up this exact (calculator, field) pair's verified
    type from CALCULATOR_FIELD_SCHEMAS. Returns None if not covered,
    in which case the caller falls back to the old global heuristics."""
    schema = CALCULATOR_FIELD_SCHEMAS.get(selected_tool) if "CALCULATOR_FIELD_SCHEMAS" in globals() else None
    if not schema:
        return None
    for f in schema:
        if f["key"] == key:
            return f  # {"type": ..., "values": [...] (if categorical), ...}
    return None


def wrap_with_unit(key, value, selected_tool=None):
    if value is None: return None

    # schema-first resolution (exact calculator + field match).
    field = _schema_field_type(selected_tool, key)
    if field is not None:
        ftype = field["type"]
        if ftype == "categorical" or ftype == "string":
            return value  # keep literal string, no coercion at all
        if ftype == "boolean":
            if isinstance(value, bool): return value
            v_str = str(value).strip().lower()
            if v_str in BOOLEAN_FALSE or v_str in ("", "nan", "unknown"):
                return False
            return True  # any other non-empty text/value = present
        if ftype == "number":
            if isinstance(value, bool): return None
            if key in SCALAR_KEYS:
                try: return float(value)
                except (ValueError, TypeError): return None
            unit = field.get("unit")
            try:
                return [float(value), unit] if unit else [float(value), ""]
            except (ValueError, TypeError):
                return [value, unit] if unit else value

    # ---- Fallback: original global heuristics (non-schema calculators) ----
    if key in SCALAR_KEYS:
        if isinstance(value, bool): return None
        try:    return float(value)
        except (ValueError, TypeError): return None

    if key in UNIT_MAP and isinstance(value, bool):
        return None

    if key in CATEGORICAL_KEYS_FALLBACK:
        return value

    if isinstance(value, bool): return value
    v_str = str(value).strip().lower()
    if v_str in BOOLEAN_TRUE:  return True
    if v_str in BOOLEAN_FALSE: return False

    if key in STRICT_BOOLEAN_KEYS_FALLBACK and v_str not in ("", "nan", "unknown", "none"):
        return True

    if key == "sex":
        return SEX_NORM.get(v_str, SEX_NORM.get(str(value).strip(), str(value).strip()))
    if isinstance(value, list): return value
    if key in UNIT_MAP:
        try:    return [float(value), UNIT_MAP[key]]
        except: return [value, UNIT_MAP[key]]
    try:    return [float(value), ""]
    except: return value


def format_inputs(calc_name, clean_json):
    registry_name = get_registry_name(calc_name)
    if not registry_name: return {}
    tool_info  = CALC_REGISTRY.get(registry_name, {})
    params_map = tool_info.get("params", {})
    clean_json_normalized = {
        k.lower().replace(" ", "_").replace("-", "_"): v
        for k, v in clean_json.items()
    }
    params = {}
    for display_key, param_key in params_map.items():
        val = clean_json.get(param_key)
        if val is None:
            val = clean_json_normalized.get(param_key.lower())
        if val is None:
            val = clean_json_normalized.get(
                display_key.lower().replace(" ", "_")
            )
        if val is None:
            matches = get_close_matches(
                param_key.lower(),
                list(clean_json_normalized.keys()),
                n=1, cutoff=0.8
            )
            if matches:
                val = clean_json_normalized[matches[0]]
        # resolve with calculator context, omit if None.
        resolved_value = wrap_with_unit(param_key, val, selected_tool=calc_name) if val is not None else None
        if resolved_value is not None:
            params[param_key] = resolved_value
    return params


EXECUTION_ERRORS = []

def execute_tool(calc_name, params):
    registry_name = get_registry_name(calc_name)
    if not registry_name:
        EXECUTION_ERRORS.append({
            "calc_name": calc_name, "registry_name": None,
            "error_type": "NOT_IN_REGISTRY", "error": "get_registry_name returned None",
            "params": params
        })
        return None

    tool_info = CALC_REGISTRY.get(registry_name, {})
    file_name = tool_info.get("file", "").replace(".py", "").replace("-", "_")
    func_name = tool_info.get("function", "")
    if not file_name or not func_name:
        EXECUTION_ERRORS.append({
            "calc_name": calc_name, "registry_name": registry_name,
            "error_type": "MISSING_FILE_OR_FUNC",
            "error": f"file={file_name!r} func={func_name!r}",
            "params": params
        })
        return None

    call_params = params
    if registry_name == "NIH Stroke Scale (NIHSS)":
        call_params = {
            k: (0 if (k in NIHSS_SUBITEM_KEYS and v is None) else v)
            for k, v in params.items()
        }

    try:
        module = importlib.import_module(file_name)
        func   = getattr(module, func_name)
        result = func(call_params)
        answer = result.get("Answer", result.get("score", None))
        if answer is not None:
            try:
                return float(str(answer).replace(",", ""))
            except (ValueError, TypeError):
                return str(answer)
        return None
    except Exception as e:
        EXECUTION_ERRORS.append({
            "calc_name": calc_name, "registry_name": registry_name,
            "error_type": type(e).__name__, "error": str(e),
            "params": call_params
        })
        return None

print("=" * 60)
print("Tool Executor ready (schema-aware types).")
print("Schema-covered calculators (exact per-field types):",
      list(CALCULATOR_FIELD_SCHEMAS.keys()) if "CALCULATOR_FIELD_SCHEMAS" in globals() else "NONE LOADED YET -- run the schema cell first!")
print("=" * 60)


In [ ]:
# ============================================================
# Create a reusable tool-selection function.
# Strategy: Zero-Shot Direct
# ============================================================

def slm_select_tool(patient_note, question):
    """
    Select the most appropriate clinical calculator tool
    using the selected model and normalize the selected tool name.
    """

    # Define the system instruction for clinical calculator selection.
    system_message = "You are a clinical calculator tool-selection assistant. Return only the calculator name."

    # Define the user message using patient note and clinical question.
    user_message = f"""
Patient Note:
{patient_note}

Question:
{question}

Select the most appropriate clinical calculator.
Return only the calculator name.
"""

    # Build chat messages.
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user",   "content": user_message}
    ]

    # Convert chat messages into model-specific prompt text.
    prompt_text = slm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    # Tokenize the prompt text.
    inputs = slm_tokenizer(prompt_text, return_tensors="pt")

    # Move tokenized inputs to the model device.
    inputs = {
        key: value.to(slm_model.device)
        for key, value in inputs.items()
    }

    # Store input token length for decoding only new tokens.
    input_token_length = inputs["input_ids"].shape[-1]

    # Generate the model response.
    outputs = slm_model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False,
        pad_token_id=slm_tokenizer.eos_token_id
    )

    # Extract only generated tokens.
    generated_tokens = outputs[0][input_token_length:]

    # Decode generated tokens into text.
    raw_response = slm_tokenizer.decode(
        generated_tokens, skip_special_tokens=True
    ).strip()

    # Normalize the raw tool response.
    normalized_tool = normalize_tool_name(raw_response)

    # Return both raw and normalized tool outputs.
    return raw_response, normalized_tool

print("=" * 60)
print("Reusable tool-selection function created successfully.")
print("=" * 60)


In [ ]:
# ============================================================
# Run full tool selection — single pass.
# Complete normalizer already defined in Section 4.
# Produces tool_selection_results_df directly.
# slm_select_tool returns (raw_output, normalized_tool).
# ============================================================

import pandas as pd

predicted_tool_outputs     = []
predicted_normalized_tools = []
ground_truth_tools         = []
is_correct_flags           = []

TOTAL_SAMPLES = len(data_df)

print("=" * 60)
print("Tool Selection Started — Zero-Shot Direct")
print("Total Samples:", TOTAL_SAMPLES)
print("=" * 60)

for sample_index in range(TOTAL_SAMPLES):

    row           = data_df.iloc[sample_index]
    patient_note  = row["Patient Note"]
    question      = row["Question"]
    gt_calculator = row["Calculator Name"]

    # Run SLM selection — returns (raw_output, normalized_tool).
    raw_output, norm_pred = slm_select_tool(patient_note, question)

    # Map GT to OMC name.
    gt_omc = TOOL_NAME_MAP.get(gt_calculator, "UNKNOWN_GT")

    # Compare both in OMC name space.
    is_correct = (norm_pred == gt_omc) and (norm_pred != "UNKNOWN_TOOL")

    predicted_tool_outputs.append(raw_output)
    predicted_normalized_tools.append(norm_pred)
    ground_truth_tools.append(gt_omc)
    is_correct_flags.append(is_correct)

    if (sample_index + 1) % 25 == 0:
        print(f"Completed: {sample_index + 1}/{TOTAL_SAMPLES}")

# Build results dataframe directly.
tool_selection_results_df = pd.DataFrame({
    "ground_truth_tool":      ground_truth_tools,
    "slm_raw_output":        predicted_tool_outputs,
    "predicted_tool":         predicted_normalized_tools,
    "updated_predicted_tool": predicted_normalized_tools,
    "is_correct":             is_correct_flags,
    "updated_is_correct":     is_correct_flags,
    "calculator_name":        data_df["Calculator Name"].tolist(),
    "category":               data_df["Category"].tolist(),
})

# Build final_results_df for compatibility.
final_results_df = tool_selection_results_df.copy()
final_results_df["predicted_tool"] = predicted_normalized_tools

print("=" * 60)
print("Tool selection completed.")
print("Correct:", sum(is_correct_flags), "/", TOTAL_SAMPLES)
print("=" * 60)


In [ ]:
# Compute calculator selection accuracy (|S| / N).


# Count total benchmark samples.
total_samples = len(tool_selection_results_df)


# Count final UNKNOWN or out-of-scope predictions.
out_of_scope_predictions = sum(
    tool_selection_results_df["updated_predicted_tool"] == "UNKNOWN_TOOL"
)


# Count valid predictions by excluding UNKNOWN or out-of-scope tools.
valid_predictions = total_samples - out_of_scope_predictions


# Count correct valid predictions.
correct_valid_predictions = sum(
    tool_selection_results_df["updated_is_correct"]
)


# Compute calculator selection accuracy (|S| / N).
selection_accuracy = (correct_valid_predictions / total_samples) * 100


# Print a separator line for readable notebook output.
print("=" * 60)

# Print evaluation title.
print("Calculator Selection Accuracy (|S| / N)")

# Print a separator line for readable notebook output.
print("=" * 60)


# Print total benchmark samples.
print("Total Samples:", total_samples)

# Print out-of-scope predictions.
print("Out-of-Scope / UNKNOWN Predictions:", out_of_scope_predictions)

# Print valid predictions.
print("Valid Predictions:", valid_predictions)

# Print correct valid predictions.
print("Correct Valid Predictions:", correct_valid_predictions)

# Print valid tool-selection accuracy.
print(
    "Valid Tool-Selection Accuracy:",
    round(selection_accuracy, 2),
    "%"
)


# Print a separator line for readable notebook output.
print("=" * 60)



In [ ]:
# Prepare extraction dataset.

print("Total Samples:", len(data_df))
print("Tool-selection results:", len(final_results_df))

if len(data_df) != len(final_results_df):
    raise ValueError("Row count mismatch.")

extraction_df_full = data_df.copy()
extraction_df_full["sample_id"] = extraction_df_full.index
extraction_df_full["slm_raw_tool_output"] = final_results_df["slm_raw_output"]

# Use updated_is_correct and updated_predicted_tool if available
if "updated_is_correct" in tool_selection_results_df.columns:
    extraction_df_full["tool_selection_correct"] = tool_selection_results_df["updated_is_correct"].values
    extraction_df_full["selected_tool"] = tool_selection_results_df["updated_predicted_tool"].values
else:
    extraction_df_full["tool_selection_correct"] = final_results_df["is_correct"]
    extraction_df_full["selected_tool"] = final_results_df["predicted_tool"]

# Filter to i in S only
extraction_df = extraction_df_full[
    extraction_df_full["tool_selection_correct"] == True
].reset_index(drop=True)

print("=" * 60)
print("N_total:", len(extraction_df_full))
print("|S| correct selections:", len(extraction_df))
print("Excluded:", len(extraction_df_full) - len(extraction_df))
print("=" * 60)

display(extraction_df.head())

In [ ]:
# ============================================================
# Section 10: Complete Entity Normalizer
# Includes:
# 1. Base normalizer functions (normalize_entity_key_final,
#    normalize_entity_value_final, normalize_extracted_entities_final,
#    values_match_final)
# 2. Semantic normalizer (SEMANTIC_ENTITY_ALIAS_MAP +
#    normalize_extracted_entities_high_accuracy)
# All in one cell — no imports needed.
# ============================================================

import re, ast, json
from difflib import get_close_matches

# ── Base normalizer maps. ─────────────────────────────────────
MEDICAL_ABBREVIATIONS = {
    "cr": "creatinine", "scr": "creatinine",
    "bun": "blood_urea_nitrogen",
    "na": "sodium", "k": "potassium",
    "cl": "chloride", "hco3": "bicarbonate",
    "co2": "bicarbonate", "hr": "heart_rate",
    "rr": "respiratory_rate", "bp": "blood_pressure",
    "sbp": "systolic_bp", "dbp": "diastolic_bp",
    "spo2": "oxygen_saturation", "hb": "hemoglobin",
    "hgb": "hemoglobin", "wbc": "white_blood_cell_count",
    "plt": "platelets", "gcs": "glasgow_coma_score",
    "inr": "inr"
}

ENTITY_ALIAS_MAP = {
    "patient_age": "age", "age_years": "age",
    "gender": "sex", "patient_gender": "sex", "patient_sex": "sex",
    "patient_weight": "weight", "body_weight": "weight", "weight_kg": "weight",
    "patient_height": "height", "height_cm": "height",
    "serum_creatinine": "creatinine", "creatinine_mg_dl": "creatinine",
    "serum_sodium": "sodium", "serum_na": "sodium",
    "serum_potassium": "potassium", "serum_k": "potassium",
    "serum_chloride": "chloride", "serum_cl": "chloride",
    "serum_bicarbonate": "bicarbonate",
    "blood_urea_nitrogen_bun": "blood_urea_nitrogen",
    "serum_glucose": "glucose", "blood_glucose": "glucose",
    "fasting_glucose": "glucose", "serum_insulin": "insulin",
    "fasting_insulin": "insulin", "serum_albumin": "albumin",
    "serum_calcium": "calcium", "patient_calcium": "calcium",
    "total_bilirubin": "bilirubin", "serum_bilirubin": "bilirubin",
    "heart_rate_or_pulse": "heart_rate", "pulse": "heart_rate",
    "patient_heart_rate": "heart_rate", "temp": "temperature",
    "o2_saturation": "oxygen_saturation", "platelet_count": "platelets",
    "white_blood_cells": "white_blood_cell_count",
    "glasgow_coma_scale": "glasgow_coma_score"
}

SCORE_ENTITY_MAP = {
    "history_of_hypertension": "hypertension_history",
    "history_of_congestive_heart_failure": "congestive_heart_failure_history",
    "history_of_chf": "congestive_heart_failure_history",
    "history_of_dvt": "previously_documented_deep_vein_thrombosis",
    "previous_dvt": "previously_documented_deep_vein_thrombosis",
    "history_of_pe": "previously_documented_pulmonary_embolism",
    "previous_pe": "previously_documented_pulmonary_embolism",
    "active_cancer": "malignancy_with_treatment_within_6_months_or_palliative",
    "malignancy": "malignancy_with_treatment_within_6_months_or_palliative",
    "recent_surgery": "surgery_in_the_previous_4_weeks",
    "immobilization": "immobilization_for_at_least_3_days",
    "clinical_signs_of_dvt": "clinical_signs_and_symptoms_of_deep_vein_thrombosis",
    "pe_most_likely": "pulmonary_embolism_is_1_diagnosis_or_equally_likely",
}

NOISE_FIELDS = {
    "calculator","tool","tool_name","name","version",
    "equation","description","source","points",
    "score","output","metadata","status"
}

# ── Base normalizer functions. ────────────────────────────────
def clean_entity_key(key):
    key = str(key).strip().lower()
    key = re.sub(r"[^a-z0-9]+", "_", key)
    key = re.sub(r"_+", "_", key).strip("_")
    return key

def normalize_entity_key_final(key):
    key = clean_entity_key(key)
    key = MEDICAL_ABBREVIATIONS.get(key, key)
    key = ENTITY_ALIAS_MAP.get(key, key)
    key = SCORE_ENTITY_MAP.get(key, key)
    return key

def normalize_boolean_value(value):
    s = str(value).strip().lower()
    if s in {"true","yes","present","positive","1","y"}: return True
    if s in {"false","no","absent","negative","0","n"}:  return False
    return value

def normalize_categorical_value(value):
    s = str(value).strip().lower()
    if s in {"m","male","man"}:    return "male"
    if s in {"f","female","woman"}: return "female"
    return value

def extract_numeric_value(value):
    if isinstance(value, list) and len(value) > 0:
        value = value[0]
    s = str(value).strip().lower()
    m = re.search(r"-?\d+\.?\d*", s)
    if m:
        try: return float(m.group(0))
        except: return value
    return value

def normalize_unit_value(value, entity_key=None):
    unit = ""
    if isinstance(value, list):
        numeric = value[0] if len(value) > 0 else value
        unit = str(value[1]).lower() if len(value) > 1 else ""
    else:
        numeric = value
    numeric = extract_numeric_value(numeric)
    if isinstance(numeric, float):
        if entity_key == "height":
            if "m" in unit and "cm" not in unit:
                return round(numeric * 100, 4)
            if numeric < 3:
                return round(numeric * 100, 4)
        return round(numeric, 4)
    return numeric

def normalize_entity_value_final(value, entity_key=None):
    value = normalize_boolean_value(value)
    value = normalize_categorical_value(value)
    value = normalize_unit_value(value, entity_key=entity_key)
    if isinstance(value, str): return value.strip().lower()
    return value

def values_match_final(true_value, predicted_value, entity_key=None):
    true_norm = normalize_entity_value_final(true_value, entity_key=entity_key)
    pred_norm = normalize_entity_value_final(predicted_value, entity_key=entity_key)
    if true_norm == pred_norm: return True
    if isinstance(true_norm, float) and isinstance(pred_norm, float):
        diff = abs(true_norm - pred_norm)
        if diff <= 0.01: return True
        if entity_key in ["height","weight"] and diff <= 0.5: return True
    return False

def remove_markdown_json(text):
    text = str(text).strip()
    text = text.replace("```json","").replace("```JSON","").replace("```","")
    return text.strip()

def extract_json_block(text):
    text = remove_markdown_json(text)
    m = re.search(r"\{.*\}", text, re.DOTALL)
    return m.group(0) if m else text

def repair_json_text(text):
    text = extract_json_block(text)
    text = re.sub(r"\bNone\b", "null", text)
    text = re.sub(r"\bTrue\b", "true", text)
    text = re.sub(r"\bFalse\b", "false", text)
    text = re.sub(r",\s*}", "}", text)
    text = re.sub(r",\s*]", "]", text)
    return text

def parse_extraction_output(text):
    repaired = repair_json_text(text)
    try: return json.loads(repaired)
    except: pass
    try: return ast.literal_eval(repaired)
    except: return {}

def collect_structured_input_list(data):
    collected = {}
    if isinstance(data, list):
        for item in data:
            if not isinstance(item, dict): continue
            if "name" in item and "value" in item:
                collected[item["name"]] = item["value"]
            elif "parameter" in item and "value" in item:
                collected[item["parameter"]] = item["value"]
    return collected

def recursive_flatten_entities(data):
    collected = {}
    if isinstance(data, dict):
        if "name" in data and "value" in data:
            return {data["name"]: data["value"]}
        if "parameter" in data and "value" in data:
            return {data["parameter"]: data["value"]}
        for key, value in data.items():
            ck = clean_entity_key(key)
            if ck in NOISE_FIELDS and not isinstance(value, (dict, list)):
                continue
            if isinstance(value, list):
                sv = collect_structured_input_list(value)
                if sv: collected.update(sv)
                else:  collected[key] = value
            elif isinstance(value, dict):
                collected.update(recursive_flatten_entities(value))
            else:
                collected[key] = value
    elif isinstance(data, list):
        collected.update(collect_structured_input_list(data))
        for item in data:
            collected.update(recursive_flatten_entities(item))
    return collected

def remove_noise_fields(entity_dict):
    return {k: v for k, v in entity_dict.items()
            if clean_entity_key(k) not in NOISE_FIELDS}

def normalize_extracted_entities_final(raw_output, selected_tool=None):
    parsed = parse_extraction_output(raw_output)
    if not isinstance(parsed, (dict, list)): return {}
    flat = recursive_flatten_entities(parsed)
    flat = remove_noise_fields(flat)
    result = {}
    seen = set()
    for key, value in flat.items():
        nk = normalize_entity_key_final(key)
        nv = normalize_entity_value_final(value, entity_key=nk)
        if nk not in seen:
            result[nk] = nv
            seen.add(nk)
    return result

# ── Semantic normalizer maps. ─────────────────────────────────
SEMANTIC_ENTITY_ALIAS_MAP = {
    "hypertension": "hypertension_history",
    "history_of_hypertension": "hypertension_history",
    "high_blood_pressure": "hypertension_history",
    "congestive_heart_failure": "congestive_heart_failure_history",
    "heart_failure": "congestive_heart_failure_history",
    "chf": "congestive_heart_failure_history",
    "diabetes": "diabetes_mellitus",
    "stroke": "stroke_history",
    "prior_stroke": "stroke_history",
    "previous_stroke": "stroke_history",
    "tia": "stroke_history",
    "thromboembolism": "stroke_history",
    "vascular_disease": "vascular_disease_history",
    "history_of_vascular_disease": "vascular_disease_history",
    "heart_rate": "heart_rate",
    "pulse": "heart_rate",
    "tachycardia": "heart_rate",
    "hemoptysis": "hemoptysis",
    "recent_surgery": "surgery_in_the_previous_4_weeks",
    "surgery": "surgery_in_the_previous_4_weeks",
    "immobilization": "immobilization_for_at_least_3_days",
    "bedridden": "immobilization_for_at_least_3_days",
    "active_cancer": "malignancy_with_treatment_within_6_months_or_palliative",
    "cancer": "malignancy_with_treatment_within_6_months_or_palliative",
    "malignancy": "malignancy_with_treatment_within_6_months_or_palliative",
    "dvt_signs": "clinical_signs_and_symptoms_of_deep_vein_thrombosis",
    "clinical_signs_of_dvt": "clinical_signs_and_symptoms_of_deep_vein_thrombosis",
    "leg_swelling": "clinical_signs_and_symptoms_of_deep_vein_thrombosis",
    "calf_swelling": "clinical_signs_and_symptoms_of_deep_vein_thrombosis",
    "previous_dvt": "previously_documented_deep_vein_thrombosis",
    "history_of_dvt": "previously_documented_deep_vein_thrombosis",
    "previous_pe": "previously_documented_pulmonary_embolism",
    "history_of_pe": "previously_documented_pulmonary_embolism",
    "pe_likely": "pulmonary_embolism_is_1_diagnosis_or_equally_likely",
    "pe_most_likely": "pulmonary_embolism_is_1_diagnosis_or_equally_likely",
    "alternative_diagnosis": "alternative_diagnosis_to_deep_vein_thrombosis_as_likely_or_more_likely",
    "localized_tenderness": "localized_tenderness_along_the_deep_venous_system",
    "entire_leg_swollen": "entire_leg_swollen",
    "major_surgery": "major_surgery_within_12_weeks",
    "collateral_veins": "collateral_nonvaricose_superficial_veins_present",
    "bun": "blood_urea_nitrogen",
    "glucose": "glucose",
    "serum_glucose": "glucose",
    "insulin": "insulin",
    "sodium": "sodium",
    "serum_sodium": "sodium",
    "na": "sodium",
    "chloride": "chloride",
    "bicarbonate": "bicarbonate",
    "hco3": "bicarbonate",
    "albumin": "albumin",
    "calcium": "calcium",
    "creatinine": "creatinine",
    "serum_creatinine": "creatinine",
}

SCORE_ABBREVIATION_MAP = {
    "ch": "congestive_heart_failure_history",
    "c": "congestive_heart_failure_history",
    "h": "hypertension_history",
    "a": "age", "a1": "age", "a2": "age",
    "d": "diabetes_mellitus",
    "s": "sex", "s2": "stroke_history",
    "v": "vascular_disease_history",
    "age_65_74": "age", "age_75_or_older": "age",
    "female_sex": "sex", "sex_category": "sex"
}

# ── Semantic normalizer functions. ────────────────────────────
SCHEMA_TOOLS = {
    "cci", "wells_criteria_dvt", "meldna", "sofa", "caprini_score",
    "has_bled_score", "ariscat_score", "nihss", "psi_score", "gad7",
}

def normalize_semantic_entity_key(key, selected_tool=None):
    key = str(key).strip().lower()
    key = re.sub(r"[^a-z0-9]+", "_", key)
    key = re.sub(r"_+", "_", key).strip("_")
    key = normalize_entity_key_final(key)

    if selected_tool in SCHEMA_TOOLS:
        # Light cleanup only -- no alias remapping for calculators with explicit schemas.
        # format_inputs() will fuzzy-match this against the real
        # calculator param keys itself.
        return key

    if key in SCORE_ABBREVIATION_MAP:
        return SCORE_ABBREVIATION_MAP[key]
    if key in SEMANTIC_ENTITY_ALIAS_MAP:
        return SEMANTIC_ENTITY_ALIAS_MAP[key]
    match = get_close_matches(
        key, list(SEMANTIC_ENTITY_ALIAS_MAP.keys()), n=1, cutoff=0.88
    )
    if match:
        return SEMANTIC_ENTITY_ALIAS_MAP[match[0]]
    return key

def normalize_extracted_entities_high_accuracy(raw_output, selected_tool=None):
    # Step 1: Apply base normalizer.
    base_entities = normalize_extracted_entities_final(
        raw_output, selected_tool=selected_tool
    )
    # Step 2: Apply semantic alignment on top (skipped for calculators with explicit schemas
    # tools -- see normalize_semantic_entity_key).
    semantic_entities = {}
    for key, value in base_entities.items():
        semantic_key = normalize_semantic_entity_key(key, selected_tool=selected_tool)
        semantic_value = normalize_entity_value_final(
            value, entity_key=semantic_key
        )
        semantic_entities[semantic_key] = semantic_value
    return semantic_entities

def values_match_high_accuracy(true_value, predicted_value, entity_key=None):
    return values_match_final(true_value, predicted_value, entity_key=entity_key)

print("=" * 60)
print("Complete Entity Normalizer Ready")
print("Base functions: normalize_entity_key_final, normalize_entity_value_final,")
print("                normalize_extracted_entities_final, values_match_final")
print("Semantic layer: normalize_extracted_entities_high_accuracy")
print("=" * 60)


In [ ]:
# ============================================================
# Registry-Guided Extraction + Normalize (v2).
# ============================================================
import pandas as pd, json

# CALCULATOR_FIELD_SCHEMAS must already be loaded (see the schema
# cell) before this cell runs.

def build_field_instructions(selected_tool):
    """Return (fields_str, detailed_instructions) for a calculator.
    Uses CALCULATOR_FIELD_SCHEMAS when available (10 calculators),
    else falls back to the generic CALC_REGISTRY params list."""
    schema = CALCULATOR_FIELD_SCHEMAS.get(selected_tool)

    if schema:
        field_names = [f["key"] for f in schema]
        fields_str = ", ".join(field_names)
        lines = []
        for f in schema:
            label = f.get("label", f["key"])
            if f["type"] == "boolean":
                lines.append(f'- "{f["key"]}" ({label}): boolean -- true if present/yes, false if absent/no or not mentioned')
            elif f["type"] == "categorical":
                values_str = " | ".join(f'"{v}"' for v in f["values"])
                lines.append(f'- "{f["key"]}" ({label}): MUST be EXACTLY one of: {values_str}')
            elif f["type"] == "string":
                values_str = " | ".join(f'"{v}"' for v in f["values"])
                lines.append(f'- "{f["key"]}" ({label}): string, one of: {values_str}')
            else:  # number
                unit_str = f' (unit: {f["unit"]})' if f.get("unit") else ""
                range_str = f' -- valid range {f["range"]}' if f.get("range") else ""
                note_str = f' [{f["note"]}]' if f.get("note") else ""
                lines.append(f'- "{f["key"]}" ({label}): number{unit_str}{range_str}{note_str}')
        detailed_instructions = "\n".join(lines)
        return fields_str, detailed_instructions

    # Fallback: original generic behavior for non-schema calculators.
    registry_name   = get_registry_name(selected_tool)
    required_params = []
    if registry_name and registry_name in CALC_REGISTRY:
        required_params = list(CALC_REGISTRY[registry_name]["params"].keys())
    fields_str = ", ".join(required_params) if required_params else "all clinical parameters"
    return fields_str, None


def slm_extract_inputs_fair(selected_tool, patient_note, question):
    fields_str, detailed_instructions = build_field_instructions(selected_tool)

    system_message = (
        "You are a clinical entity extraction assistant. "
        "Extract only the clinical input parameters required "
        "for the selected calculator. "
        "Return only a valid flat JSON object. "
        "Do not calculate scores. "
        "Do not include explanations. "
        "Do not include scoring criteria."
    )

    if detailed_instructions:
        user_message = f"""
Selected Calculator Tool:
{selected_tool}

Required Parameters (with exact type/format for each):
{detailed_instructions}

Patient Note:
{patient_note}

Question:
{question}

Task:
Extract ALL of the parameters listed above, using their EXACT
key names and EXACT accepted values where specified.
Rules:
1. Return only valid JSON.
2. Return only flat JSON -- keys must match the parameter names above exactly.
3. For categorical fields, output ONLY one of the listed exact strings.
4. For boolean fields, output true or false (not "yes"/"no").
5. If a parameter is not found or not mentioned, set it to null.
6. Do not calculate the final score.
7. Do not include extra metadata.

Example Output:
{{
    "age": 65,
    "sex": "male",
    "weight": 70,
    "creatinine": 1.4
}}
"""
    else:
        user_message = f"""
Selected Calculator Tool:
{selected_tool}
Required Parameters for this calculator:
{fields_str}
Patient Note:
{patient_note}
Question:
{question}
Task:
Extract ONLY these required parameters: {fields_str}
Rules:
1. Return only valid JSON.
2. Return only flat JSON.
3. Extract ALL required parameters listed above.
4. If a parameter is not found, set it to null.
5. Do not calculate the final score.
6. Do not include extra metadata.
Example Output:
{{
    "age": 65,
    "sex": "male",
    "weight": 70,
    "creatinine": 1.4
}}
"""

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user",   "content": user_message}
    ]
    prompt_text = slm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = slm_tokenizer(prompt_text, return_tensors="pt")
    inputs = {k: v.to(slm_model.device) for k, v in inputs.items()}
    input_token_length = inputs["input_ids"].shape[-1]
    outputs = slm_model.generate(
        **inputs,
        max_new_tokens=300,   # slightly higher -- detailed schemas can need more room
        do_sample=False,
        pad_token_id=slm_tokenizer.eos_token_id
    )
    return slm_tokenizer.decode(
        outputs[0][input_token_length:], skip_special_tokens=True
    ).strip()


# Run extraction + normalize.
normalized_results = []
TOTAL_SAMPLES = len(extraction_df)
print("=" * 60)
print("Registry-Guided Extraction + Normalization Started (v2 -- schema-aware)")
print("Total Samples (i in S):", TOTAL_SAMPLES)
print("Schema-covered calculators:", list(CALCULATOR_FIELD_SCHEMAS.keys()))
print("=" * 60)

for sample_index in range(TOTAL_SAMPLES):
    sample_row    = extraction_df.iloc[sample_index]
    patient_note  = sample_row["Patient Note"]
    question      = sample_row["Question"]
    selected_tool = sample_row["selected_tool"]
    sample_id     = int(sample_row["sample_id"])

    if sample_index % 10 == 0:
        torch.cuda.empty_cache()

    raw_output = slm_extract_inputs_fair(
        selected_tool, patient_note, question
    )
    normalized = normalize_extracted_entities_high_accuracy(
        raw_output, selected_tool=selected_tool
    )
    normalized_results.append({
        "sample_id":                     sample_id,
        "selected_tool":                 selected_tool,
        "raw_extracted_output":          raw_output,
        "normalized_extracted_entities": json.dumps(normalized)
    })
    if (sample_index + 1) % 25 == 0:
        print(f"Completed: {sample_index + 1}/{TOTAL_SAMPLES}")

fair_normalized_df       = pd.DataFrame(normalized_results)
normalized_extraction_df = fair_normalized_df.copy()
fair_normalized_df.to_csv(
    os.path.join(RESULTS_DIR, "slm_fair_semantic_normalized_results.csv"),
    index=False
)
print("=" * 60)
print("Registry-Guided Extraction completed (v2 -- schema-aware).")
print("Shape:", fair_normalized_df.shape)
print("=" * 60)
display(fair_normalized_df.head())


In [ ]:
# ============================================================
# PEA — Extraction Accuracy (fair_normalized_df)
# Formula: matched_fields / total_fields x 100
# Source: fair_normalized_df (extraction + entity normalizer)
# ============================================================

import ast, json, re

matched_fields = 0
total_fields   = 0
parsed_samples = 0
failed_samples = 0

for sample_index in range(len(fair_normalized_df)):

    selected_tool = fair_normalized_df.iloc[sample_index]["selected_tool"]

    # Parse GT entities.
    try:
        gt_entities = ast.literal_eval(
            extraction_df.iloc[sample_index]["Relevant Entities"]
        )
    except:
        failed_samples += 1
        continue

    # Parse predicted entities.
    try:
        predicted_entities = json.loads(
            fair_normalized_df.iloc[sample_index]["normalized_extracted_entities"]
        )
    except:
        failed_samples += 1
        continue

    if len(predicted_entities) == 0:
        failed_samples += 1
        continue

    parsed_samples += 1

    for gt_key, gt_value in gt_entities.items():
        total_fields += 1
        normalized_gt_key = normalize_semantic_entity_key(gt_key, selected_tool=selected_tool)
        if normalized_gt_key not in predicted_entities:
            continue
        if values_match_high_accuracy(
            gt_value,
            predicted_entities[normalized_gt_key],
            entity_key=normalized_gt_key
        ):
            matched_fields += 1

# Compute PEA accuracy.
fair_semantic_accuracy = (matched_fields / total_fields) * 100

print("=" * 60)
print("PEA: Extraction Accuracy")
print("=" * 60)
print("Parsed Samples:", parsed_samples)
print("Failed Samples:", failed_samples)
print("Matched Fields:", matched_fields)
print("Total Fields:  ", total_fields)
print("PEA Accuracy:   ", round(fair_semantic_accuracy, 2), "%")
print("=" * 60)


In [ ]:
# ============================================================
# Tool Calling Computation — Agent 3 + Agent 4.
# Computation via .py file execution — NOT SLM prompt.
# ============================================================
import re as _re, json as _json, ast as _ast

def slm_compute_clinical_result(selected_tool, extracted_inputs, question=None):
    """
    Step 1 — Parse raw extraction JSON.
    Step 2 — Agent 3: format_inputs() wraps values with units.
    Step 3 — Agent 4: execute_tool() runs .py calculator file.
    """
    clean_json = {}
    try:
        if isinstance(extracted_inputs, str):
            clean = extracted_inputs.strip()
            clean = _re.sub(r"```json\s*","", clean, flags=_re.IGNORECASE)
            clean = _re.sub(r"```\s*","", clean).strip()
            clean = _re.sub(r"\bNone\b","null", clean)
            clean = _re.sub(r"\bTrue\b","true", clean)
            clean = _re.sub(r"\bFalse\b","false", clean)
            clean = _re.sub(r",\s*}","}", clean)
            open_b  = clean.count("{")
            close_b = clean.count("}")
            if open_b > close_b:
                clean += "}" * (open_b - close_b)
            try:
                clean_json = _json.loads(clean)
            except:
                m = _re.search(r"\{.*\}", clean, _re.DOTALL)
                if m:
                    try:    clean_json = _json.loads(m.group())
                    except:
                        try: clean_json = _ast.literal_eval(m.group())
                        except: clean_json = {}
        else:
            clean_json = extracted_inputs if isinstance(extracted_inputs, dict) else {}
    except:
        clean_json = {}

    if not clean_json:
        return "None"

    params = format_inputs(selected_tool, clean_json)
    result = execute_tool(selected_tool, params)

    return str(result) if result is not None else "None"

print("=" * 60)
print("Tool Calling computation function ready.")
print("Agent 3 (format_inputs) + Agent 4 (execute_tool) active.")
print("=" * 60)


In [ ]:
# Run Tool Calling computation on all samples in S.
import pandas as pd

full_computation_results = []
TOTAL_SAMPLES = len(fair_normalized_df)

print("=" * 60)
print("Tool Calling Computation Started")
print("Total Samples (i in S):", TOTAL_SAMPLES)
print("=" * 60)

for sample_index in range(TOTAL_SAMPLES):
    sample_row    = extraction_df.iloc[sample_index]
    selected_tool = sample_row["selected_tool"]
    question      = sample_row["Question"]
    sample_id     = int(sample_row["sample_id"])

    extracted_inputs = fair_normalized_df.iloc[sample_index]["normalized_extracted_entities"]

    if sample_index % 10 == 0:
        torch.cuda.empty_cache()

    predicted_output = slm_compute_clinical_result(
        selected_tool, extracted_inputs, question
    )

    full_computation_results.append({
        "sample_id":           sample_id,
        "selected_tool":       selected_tool,
        "extracted_inputs":    extracted_inputs,
        "slm_computed_answer": predicted_output,
        "ground_truth_answer": sample_row["Ground Truth Answer"],
        "lower_limit":         sample_row["Lower Limit"],
        "upper_limit":         sample_row["Upper Limit"]
    })

    if (sample_index + 1) % 25 == 0:
        print(f"Completed: {sample_index + 1}/{TOTAL_SAMPLES}")

full_computation_df = pd.DataFrame(full_computation_results)

full_computation_df.to_csv(
    os.path.join(RESULTS_DIR, "slm_full_slm_computation_results.csv"),
    index=False
)

if EXECUTION_ERRORS:
    pd.DataFrame(EXECUTION_ERRORS).to_csv(
        os.path.join(RESULTS_DIR, "execution_errors.csv"), index=False
    )

print("=" * 60)
print("Tool Calling computation completed.")
print("Shape:", full_computation_df.shape)
print("Execution errors captured:", len(EXECUTION_ERRORS),
      "(saved to execution_errors.csv)" if EXECUTION_ERRORS else "")
print("=" * 60)
display(full_computation_df.head())


In [ ]:
# ============================================================
# Computation Accuracy
# Formula: |C| / |S| x 100
# Denominator = correct selections only
# ============================================================
import re

def extract_numeric_value(text):
    match = re.search(r"-?\d+\.?\d*", str(text))
    return float(match.group(0)) if match else None

# S = correct selections = extraction_df
S_count = len(extraction_df)

# Count correct computations.
correct_predictions = 0
comp_correct_flags  = {}

for sample_index in range(len(full_computation_df)):
    sid   = int(full_computation_df.iloc[sample_index]["sample_id"])
    pred  = extract_numeric_value(full_computation_df.iloc[sample_index]["slm_computed_answer"])
    lower = extract_numeric_value(full_computation_df.iloc[sample_index]["lower_limit"])
    upper = extract_numeric_value(full_computation_df.iloc[sample_index]["upper_limit"])

    if lower is None or upper is None or pred is None:
        comp_correct_flags[sid] = False
        continue

    is_corr = (lower <= pred <= upper)
    comp_correct_flags[sid] = is_corr
    if is_corr:
        correct_predictions += 1

# CCA = |C| / |S|
slm_computation_accuracy = (
    correct_predictions / S_count * 100
) if S_count > 0 else 0

print("=" * 60)
print("CCA: Computation Accuracy")
print("Formula: |C| / |S| x 100")
print("=" * 60)
print("|S| correct selections :", S_count)
print("|C| correct computations:", correct_predictions)
print("CCA Accuracy:", round(slm_computation_accuracy, 2), "%")
print("=" * 60)

## Error analysis, category-wise evaluation and summary

The next three cells are identical in all notebooks except for the first one, which collects the per-sample results of this pipeline.

In [ ]:
# Per-sample evaluation table (inputs of the error analysis and of the category-wise evaluation).
import ast, json
import pandas as pd

sel = tool_selection_results_df["updated_is_correct"].astype(bool).tolist()
ext_info, ext_unmatched = {}, []
for row_i in range(len(fair_normalized_df)):
    sid = int(extraction_df.iloc[row_i]["sample_id"])
    try:
        gt_ent = ast.literal_eval(extraction_df.iloc[row_i]["Relevant Entities"])
        pred_ent = json.loads(fair_normalized_df.iloc[row_i]["normalized_extracted_entities"])
    except Exception:
        continue                      # unparsable sample: not scored (as in PEA)
    if len(pred_ent) == 0:
        continue
    matched = 0
    for gt_key, gt_val in gt_ent.items():
        norm_key = normalize_semantic_entity_key(gt_key, selected_tool=fair_normalized_df.iloc[row_i]["selected_tool"])
        if norm_key in pred_ent and values_match_high_accuracy(gt_val, pred_ent[norm_key], entity_key=norm_key):
            matched += 1
        else:
            ext_unmatched.append({"sample_id": sid, "category": data_df.iloc[sid]["Category"], "key": norm_key,
                                  "gt": gt_val, "pred": pred_ent.get(norm_key)})
    ext_info[sid] = (len(gt_ent), matched)

comp_info, comp_wrong = {}, []
for row_i in range(len(full_computation_df)):
    sid = int(full_computation_df.iloc[row_i]["sample_id"])
    ok = bool(comp_correct_flags.get(sid, False))
    comp_info[sid] = ok
    if not ok:
        comp_wrong.append({"sample_id": sid, "category": data_df.iloc[sid]["Category"],
                           "pred": full_computation_df.iloc[row_i]["slm_computed_answer"],
                           "gt": full_computation_df.iloc[row_i]["ground_truth_answer"]})
eval_df = pd.DataFrame([{
    "sample_id": i, "category": data_df.iloc[i]["Category"], "calculator": data_df.iloc[i]["Calculator Name"],
    "selected_correct": bool(sel[i]),
    "ext_evaluated": i in ext_info, "n_entities": ext_info.get(i, (0, 0))[0], "n_matched": ext_info.get(i, (0, 0))[1],
    "comp_evaluated": i in comp_info, "comp_correct": bool(comp_info.get(i, False)),
} for i in range(len(data_df))])

In [ ]:
# Error taxonomy: rule-based operationalization of the categories of Wang et al. (2025), "From Scores to Steps".
# The same rules are used in all six notebooks.
import re

ERROR_TYPES = [
    "formula_misselection_hallucination",  # selected calculator differs from the ground-truth calculator
    "incorrect_variable_extraction",       # wrong value of a numeric variable (or date)
    "missing_variable",                    # required variable absent from the extraction
    "clinical_misinterpretation",          # wrong boolean / categorical value (presence, severity, threshold)
    "demographic_adjustment_failure",      # wrong or missing age, sex, race, BSA or pregnancy variable
    "arithmetic_error",                    # formula and inputs right, result wrong beyond rounding
    "unit_conversion_error",               # value differs from the ground truth by a unit-conversion factor
    "precision_error",                     # result within the rounding tolerance of the ground truth but outside the dataset interval
]
DEMOGRAPHIC_TOKENS = {"age", "sex", "gender", "race", "ethnicity", "pregnancy", "pregnant", "bsa"}
UNIT_FACTORS = [2.20462, 0.45359, 2.54, 0.3937, 88.4, 0.01131, 18.0182, 0.0555, 2.8, 0.357,
                38.67, 0.02586, 4.008, 0.2495, 17.1, 0.0585, 10.0, 0.1, 1000.0, 0.001]
UNIT_FACTOR_TOLERANCE = 0.03    # relative tolerance when comparing the value ratio with a conversion factor
ROUNDING_TOLERANCE = {1: 0.05, 2: 0.005}   # Wang et al.: +-0.05 for one decimal place, +-0.005 for two
DATE_PATTERN = re.compile(r"^\s*\d{1,2}/\d{1,2}/\d{2,4}")

def scalar(v):
    return v[0] if isinstance(v, (list, tuple)) and len(v) > 0 else v

def to_float(v):
    v = scalar(v)
    if v is None or isinstance(v, bool):
        return None
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).strip().replace(",", "")
    if DATE_PATTERN.match(s):
        return None
    m = re.search(r"-?\d+\.?\d*", s)
    return float(m.group(0)) if m else None

def is_missing(v):
    return v is None or str(v).strip().lower() in ("", "none", "null", "nan")

def is_demographic_key(key):
    tokens = set(re.split(r"[^a-z0-9]+", str(key).lower()))
    return bool(tokens & DEMOGRAPHIC_TOKENS) or "body_surface" in str(key).lower() or "surface area" in str(key).lower()

def is_unit_error(gt, pred):
    g, p = to_float(gt), to_float(pred)
    if g is None or p is None or g == 0 or p == 0:
        return False
    ratio = p / g
    if any(abs(ratio / f - 1) <= UNIT_FACTOR_TOLERANCE for f in UNIT_FACTORS):
        return True
    return abs(p - (g * 9 / 5 + 32)) <= 1 or abs(p - (g - 32) * 5 / 9) <= 1   # Celsius <-> Fahrenheit

def classify_extraction_error(key, gt, pred):
    if is_missing(scalar(pred)):
        return "missing_variable"
    if is_demographic_key(key):
        return "demographic_adjustment_failure"
    if is_unit_error(gt, pred):
        return "unit_conversion_error"
    g = scalar(gt)
    if to_float(g) is None and not DATE_PATTERN.match(str(g)):
        return "clinical_misinterpretation"
    return "incorrect_variable_extraction"

def decimals(text):
    s = str(scalar(text)).strip()
    m = re.search(r"\.(\d+)", s)
    return min(len(m.group(1)), 2) if m else 0

def classify_computation_error(pred, gt):
    p, g = to_float(pred), to_float(gt)
    if p is None or g is None:
        return "arithmetic_error"
    tol = ROUNDING_TOLERANCE.get(decimals(pred), ROUNDING_TOLERANCE[1] if decimals(pred) == 0 else 0.005)
    return "precision_error" if abs(p - g) <= tol else "arithmetic_error"

In [ ]:
# Error analysis, category-wise evaluation and overall summary (identical in all notebooks).
# Inputs built by the previous cell: eval_df (one row per sample), ext_unmatched, comp_wrong.
import json, os
import pandas as pd

N = len(eval_df)
n_selected = int(eval_df["selected_correct"].sum())
ext_part = eval_df[eval_df["ext_evaluated"]]
comp_part = eval_df[eval_df["comp_evaluated"]]

def pct(a, b):
    return round(a / b * 100, 2) if b else float("nan")

# ---- overall metrics
CSA = pct(n_selected, N)                                   # |S| / N
PEA = pct(int(ext_part["n_matched"].sum()), int(ext_part["n_entities"].sum()))   # matched / ground-truth entities
CCA = pct(int(comp_part["comp_correct"].sum()), len(comp_part))                  # |C| / |D|

print("=" * 70)
print(f"CSA = |S| / N                : {n_selected} / {N} = {CSA} %")
print(f"PEA = matched / entities     : {int(ext_part['n_matched'].sum())} / {int(ext_part['n_entities'].sum())} = {PEA} %")
print(f"CCA = |C| / |D|              : {int(comp_part['comp_correct'].sum())} / {len(comp_part)} = {CCA} %")
print("=" * 70)

# ---- error analysis
error_counts = {t: 0 for t in ERROR_TYPES}
error_records = []
for _, r in eval_df[~eval_df["selected_correct"]].iterrows():
    error_counts["formula_misselection_hallucination"] += 1
    error_records.append({"sample_id": r["sample_id"], "category": r["category"], "stage": "selection",
                          "error_type": "formula_misselection_hallucination"})
for e in ext_unmatched:
    t = classify_extraction_error(e["key"], e["gt"], e["pred"])
    error_counts[t] += 1
    error_records.append({"sample_id": e["sample_id"], "category": e["category"], "stage": "extraction",
                          "field": e["key"], "error_type": t})
for w in comp_wrong:
    t = classify_computation_error(w["pred"], w["gt"])
    error_counts[t] += 1
    error_records.append({"sample_id": w["sample_id"], "category": w["category"], "stage": "computation",
                          "error_type": t})

total_errors = sum(error_counts.values())
print(f'{"Error type":<40} {"Count":>7} {"Share":>8}')
for t in ERROR_TYPES:
    print(f"{t:<40} {error_counts[t]:>7} {pct(error_counts[t], total_errors):>7}%")
print(f'{"TOTAL":<40} {total_errors:>7}')
print("(formula errors and computation errors are counted per sample, extraction errors per unmatched entity)")

# ---- category-wise evaluation (same formulas as above, restricted to one category)
def stage_table(group_col):
    rows = []
    for g in sorted(eval_df[group_col].dropna().unique(), key=str):
        d = eval_df[eval_df[group_col] == g]
        e = d[d["ext_evaluated"]]
        c = d[d["comp_evaluated"]]
        rows.append({group_col: g, "N": len(d), "S": int(d["selected_correct"].sum()),
                     "CSA_%": pct(int(d["selected_correct"].sum()), len(d)),
                     "PEA_%": pct(int(e["n_matched"].sum()), int(e["n_entities"].sum())),
                     "CCA_%": pct(int(c["comp_correct"].sum()), len(c)),
                     "D": len(c)})
    return pd.DataFrame(rows)

cat_df = stage_table("category")
print(cat_df.to_string(index=False))
if "EQUATION_BASED_CALCS" in globals():
    eval_df["nature"] = eval_df["calculator"].apply(lambda c: "Equation" if c in EQUATION_BASED_CALCS else "Rule-based")
    nature_df = stage_table("nature")
    print(nature_df.to_string(index=False))
    nature_df.to_csv(os.path.join(RESULTS_DIR, "nature_wise_results.csv"), index=False)

# ---- save
os.makedirs(RESULTS_DIR, exist_ok=True)
pd.DataFrame(error_records).to_csv(os.path.join(RESULTS_DIR, "error_analysis.csv"), index=False)
with open(os.path.join(RESULTS_DIR, "error_summary.json"), "w") as f:
    json.dump({"error_counts": error_counts, "total_errors": total_errors}, f, indent=4)
cat_df.to_csv(os.path.join(RESULTS_DIR, "category_results.csv"), index=False)
eval_df.to_csv(os.path.join(RESULTS_DIR, "per_sample_evaluation.csv"), index=False)
summary = {"dataset": DATASET_NAME, "model": MODEL_ID, "N": N,
           "S": n_selected, "CSA_%": CSA,
           "PEA_matched": int(ext_part["n_matched"].sum()), "PEA_entities": int(ext_part["n_entities"].sum()), "PEA_%": PEA,
           "CCA_correct": int(comp_part["comp_correct"].sum()), "CCA_denominator": len(comp_part), "CCA_%": CCA}
if "technical_errors" in globals():
    summary["technical_errors"] = int(technical_errors)   # MCP only
with open(os.path.join(RESULTS_DIR, "overall_summary.json"), "w") as f:
    json.dump(summary, f, indent=4)
pd.DataFrame([summary]).to_csv(os.path.join(RESULTS_DIR, "All_combined_results.csv"), index=False)
print("Results saved in", RESULTS_DIR)

In [ ]:
# ── Define S_indices first ──
S_indices = set(
    tool_selection_results_df[
        tool_selection_results_df["updated_is_correct"] == True
    ].index.tolist()
)
print("S_indices count:", len(S_indices))